# Clinical knowledge-informed neural networks (CKINN) for type 2 diabetes risk: development in RICH, frozen external evaluation in NAGALA, and NHANES screening

This notebook reproduces every analysis of the study from the original data files. It is self-contained: the cells below write the analysis package (`ckinn_obj1/`), verify the inputs by SHA-256, and run the complete pipeline. Code cells contain code only. The text cells explain each step.

## Study design

| Element | Definition |
|---|---|
| Task A (primary) | Recorded incident type 2 diabetes within three years of a baseline health examination |
| Development cohort | Rich Healthcare Group, China (RICH; Chen et al., BMJ Open 2018; Dryad 10.5061/dryad.ft8750v) |
| External cohort | NAGALA, Japan (Okamura et al., Int J Obes 2019; Dryad 10.5061/dryad.8q0p192), scored only with models frozen in RICH |
| Eligibility | Age ≥ 20 years, positive follow-up, binary event, baseline FPG < 6.1 mmol/L (NAGALA additionally HbA1c < 6.5%) |
| Predictors | Age, sex, BMI, systolic blood pressure, ALT, fasting plasma glucose, triglycerides, HDL cholesterol (plus missingness indicators) |
| Censoring | Inverse probability of censoring weights from a reverse Kaplan–Meier estimate; cases 1/G(T−), controls 1/G(3), unknown status 0 |
| Models | M1 unconstrained branched network; M2 = M1 with explanations; M3 = CKINN (M1 plus a soft penalty on wrong-signed probability gradients, FPG+ and BMI+, λ tuned in {0, 1, 10, 100}); M4 site-balanced; M5 sex loss-gap penalty; logistic regression, random forest, XGBoost, LightGBM, linear SVM, ridge Cox, intercept-only reference |
| Primary contrast | Brier(M3) − Brier(M1) in NAGALA with a paired participant bootstrap (censoring weights re-estimated per draw) |
| Secondary contrast | Matched positive versus zero penalty with identical architecture, preprocessing, splits, seeds and epochs |
| Task B (separate) | Elevated HbA1c (≥ 6.5%) at the examination among NHANES participants without a reported diagnosis; trained on 2017–2020, frozen and evaluated on 2021–2023, four non-glycaemic inputs |

Tuning, early stopping, penalty selection and thresholds use RICH development partitions only. NAGALA and the later NHANES period are never used for fitting.

## Proposal coverage (CKINN_OBJ, sections 22–24)

| Proposal item | Where it is answered | Output |
|---|---|---|
| RQ1 Predict future type 2 diabetes from baseline variables | Stages 21–25 | `pooled_oof.csv`, `external_metrics.csv` |
| RQ2 / H1 Clinical knowledge improves prediction over an unconstrained network | Stage 26 (primary contrast), stage 28, EXT E9–E10 | `paired_brier.csv`, `matched_positive_paired_brier.csv`, `ablations.csv` |
| RQ3 / H3 Clinical knowledge improves cross-population generalisation | Stages 24–26 (RICH → NAGALA), stage 29 (held-out RICH sites), EXT E6, E9 | `external_metrics.csv`, `A_RICH_site.csv`, `EXT/*` |
| RQ4 / H2 Calibration is maintained externally and improved by constraints | Stages 25–27, EXT E12 | `calibration_bins.csv`, `calibration_roc_pr.png`, `EXT/E12_calibration_contrast.csv` |
| RQ5 / H4 Explanations are clinically consistent | Stage 30 (integrated gradients, random-forest SHAP, finite-step direction checks, ICE), EXT E2, E8 | `directions.csv`, `attributions.csv`, `rf_shap.csv`, `EXT/E2`, `EXT/E8` |
| RQ6 / H5 Performance is consistent across demographic groups | Stages 28, 31 (sex, age, site, missingness; M1, M3, M4, M5) | `subgroups.csv`, `sex_gaps.csv` |
| RQ7 Elevated risk before diagnostic thresholds | Stage 31 (baseline FPG < 5.6 mmol/L) | `subgroups.csv` |
| Section 24 evaluation framework | AUROC, average precision, sensitivity, specificity, PPV, NPV, F1, FPR, FNR, Brier, calibration intercept and slope, calibration plot, external and cross-population validation, group metrics, attribution and direction checks, net benefit and IPA (EXT) | All tables |
| Section 3A, current status | Task B, stages 33–35 | `B_as_coded/*` |
| Sections 3C, 11, 26 and clinical expert agreement | Not measurable with these data (no repeated dated states, glucose–insulin dynamics, curated knowledge graph or clinician panel) | Listed in `proposal_hypotheses_evidence.csv` |

H1 is the single pre-specified confirmatory contrast. All other comparisons are descriptive. Analyses labelled EXT were specified after the first full development run and are exploratory. Their rules are in `EXT_PRESPECIFICATION.md`.

## How to run

1. Place the original files in one folder (`data/raw` under the project folder): `RCHealthCareData-20180820.xlsx` (RICH), `DatasetIntJObesity.xlsx` (NAGALA) and `Diabetes_Health_Indicators_NHANES_2017-2023.zip` (NHANES extract). File names may differ; the files are identified by their SHA-256.
2. In Colab, choose a CPU runtime and run all cells. Outside Colab, set `CKINN_ROOT` to the project folder.
3. `RUN_MODE`: `SMOKE` checks the workflow on small samples in minutes and its numbers must not be interpreted. `STANDARD` is the reported analysis (5 × 2 nested cross-validation, 6 network trials, 100 epochs with early stopping, 200 bootstrap draws for the v2 contrasts and 1,000 for EXT). `FULL` adds three network seeds, 12 trials and 1,000 bootstrap draws.
4. Every fit is checkpointed. After an interruption, rerun the notebook. To run one part, set `RUN_STAGE` to `AUDIT`, `CHECKS`, `INTERNAL`, `FINAL`, `EXTERNAL`, `ABLATIONS`, `SENSITIVITY`, `EXPLAIN`, `SUBGROUPS`, `UPDATE`, `SCREEN`, `EXT` or `REPORT`.
5. STANDARD needs about 20 CPU hours in total (stage 29 is about half). Results are written to `results/`. Stage 39 writes an aggregate ZIP without participant-level data.

## Integrity

Only the original cohort data are analysed; no synthetic, simulated or augmented records enter any analysis. Small artificial arrays are used only inside the software checks (stages 14, 17 and 20). Every number in the output tables is computed by the code from the data. Null and unfavourable results are reported as found. Participant-level data, models and predictions stay in the private `checkpoints/` folder and are never exported.

## 01 · Paths and run settings

The project folder holds `data/raw` (inputs), `checkpoints/` (private models and predictions), `results/` (aggregate tables, figures and reports) and `logs/`. The run identity binds the input bytes, the package code, the configuration, the device and the software versions. Changing any of them starts a new run. Changing `RUN_STAGE` does not.

In [ ]:
import os
import sys
import json
from pathlib import Path

try:
    from google.colab import drive

    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/CKINN_OBJ1")
except ImportError:
    ROOT = Path(os.environ.get("CKINN_ROOT", "CKINN_OBJ1")).resolve()

RAW_DIR = ROOT / "data" / "raw"
PACKAGE = Path("ckinn_obj1").resolve()
PACKAGE.mkdir(parents=True, exist_ok=True)
RUN_MODE = os.environ.get("CKINN_RUN_MODE", "STANDARD")
RUN_STAGE = os.environ.get("CKINN_RUN_STAGE", "ALL")
DEVICE = "cpu"
OPTION_OVERRIDES = {"broader_rich": True, "site_holdout": True, "alt_zero": True, "tree_shap": True}
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
print("Project folder:", ROOT)
print("Input folder:", RAW_DIR, "| exists:", RAW_DIR.is_dir())
print("Mode:", RUN_MODE, "| stage:", RUN_STAGE, "| device:", DEVICE)

## 02 · Package module `data.py`: inputs, harmonisation and eligibility

Identifies the input files by SHA-256 and reads the workbooks, including both cached values and formula text. It harmonises names, units and sex codes (male = 1) and sets non-positive physical measurements to missing. It applies the eligibility rules step by step and audits the NHANES extract. The three-year status function codes a case (event by year 3), a control (follow-up beyond 3 years) or unknown status. Every predictor passes through a task-specific allowlist.

In [ ]:
%%writefile ckinn_obj1/data.py
from pathlib import Path, PurePosixPath
from hashlib import sha256
import hashlib, json, os, pickle, re, shutil, stat, tempfile, zipfile
from collections import Counter
import numpy as np
import pandas as pd
import openpyxl

HASH = {
    "RICH": "ffd341a1a91ac65585b9fba70847a2681f5671a6035d9cfbf92e50795d74fbf6",
    "NAGALA": "671359aea2019c848f969395f88e57f888eaba8463ea0351bcda8f1c947f5523",
    "NHANES": "3af63d0d53d8f88c38a7f54d80e6e448014322f7c8c3fe115ba9b3e47c2c6acf",
    "proposal": "fc5f4d618ba1bb0d33d8d44f541846e75699268a24e1c851fbee275c00307b10",
}
CORE = ["age", "sex", "bmi", "sbp", "alt", "fpg", "tg", "hdl"]
NONLAB = CORE[:4]
PLUS = CORE + ["dbp", "smoking", "tc", "ast"]
UNITS = dict(
    age="years",
    sex="category",
    bmi="kg/m²",
    sbp="mmHg",
    dbp="mmHg",
    alt="U/L",
    ast="U/L",
    fpg="mmol/L",
    tg="mmol/L",
    hdl="mmol/L",
    tc="mmol/L",
    smoking="current 1, other 0",
    a1c="%",
    time="years",
    event="binary",
)
R = dict(
    id="id",
    site="site",
    age="Age (y)",
    sex="Gender(1, male; 2, female)",
    bmi="BMI(kg/m2)",
    sbp="SBP(mmHg)",
    dbp="DBP(mmHg)",
    alt="ALT(U/L)",
    ast="AST(U/L)",
    fpg="FPG (mmol/L)",
    tc="Cholesterol(mmol/L)",
    tg="Triglyceride(mmol/L)",
    hdl="HDL-c(mmol/L)",
    event="censor of diabetes at followup(1, Yes; 0, No)",
    time="year of followup",
    smoking="smoking status(1,current smoker;2, ever smoker;3,never smoker)",
)
J = dict(
    id="Patient Number",
    age="Baseline Age, yrs",
    sex="Sex, 1:Men, 2:Women",
    bmi="Baseline BMI, kg/m2",
    sbp="Baseline Systolic blood pressure, mmHg",
    dbp="Baseline Diastolic blood pressure, mmHg",
    alt="Baseline ALT, IU/L",
    ast="Baseline AST, IU/L",
    fpg="Baseline Fasting plasma glucose, mmol/L",
    tc="Baseline Total Cholesterol, mmol/L",
    tg="Baseline Triglycerides, mmol/L",
    hdl="Baseline HDL-cholesterol, mmol/L",
    event="Incident DM 0/1",
    time="Follow up duration, days",
    smoking="Baseline Smoking status, 1:never, 2:past, 3:current",
    a1c="Baseline HbA1c, %",
)
PERIODS = ["2017-2020", "2021-2023"]
NHC = "Diabetes_self HighBP HighChol BMI Smoker Stroke HeartDiseaseorAttack PhysActivity AnyHealthcare GenHlth MentHlth Sex Age AgeGroup Education Income SystolicBP DiastolicBP HbA1c FastingGlucose Diabetes_lab".split()
NH = dict(age="Age", sex="Sex", bmi="BMI", sbp="SystolicBP", a1c="HbA1c", reported="Diabetes_self")
CONVERT = {
    "hdl": ("Baseline HDL-cholesterol, mg/dL", 0.02586),
    "tc": ("Baseline Total Cholesterol, mg/dL", 0.02586),
    "tg": ("Baseline Triglycerides, mg/dL", 0.01129),
    "fpg": ("Baseline Fasting plasma glucose, mg/dl", 0.05551),
}


def digest(path):
    with open(path, "rb") as f:
        h = sha256()
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def key(obj):
    return sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()


def atomic(path, writer):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, name = tempfile.mkstemp(dir=path.parent, prefix="." + path.name)
    os.close(fd)
    try:
        writer(Path(name))
        os.replace(name, path)
    finally:
        Path(name).unlink(missing_ok=True)


def js(path, obj):
    atomic(path, lambda p: p.write_text(json.dumps(obj, indent=2, default=str), encoding="utf8"))


def csv(path, rows):
    frame = pd.DataFrame(rows).copy()
    parts = Path(path).parts
    for profile in ("SMOKE", "STANDARD", "FULL", "AUDIT"):
        if profile in parts and "results" in parts:
            frame["profile"] = profile
            frame["evidence_use"] = (
                "software check only" if profile in ("SMOKE", "AUDIT") else "scientific reanalysis"
            )
            break
    atomic(path, lambda p: frame.to_csv(p, index=False))


def dump(path, obj):
    payload = pickle.dumps(obj, protocol=5)
    atomic(path, lambda p: p.write_bytes(b"CKOBJ1 " + sha256(payload).hexdigest().encode() + b"\n" + payload))


def restore(path):
    packed = Path(path).read_bytes()
    head, sep, payload = packed.partition(b"\n")
    if not sep or head != b"CKOBJ1 " + sha256(payload).hexdigest().encode():
        raise ValueError(f"Integrity failure: {path}")
    return pickle.loads(payload)


def discover(raw):
    found = {}
    for path in sorted(Path(raw).iterdir()):
        if path.is_file() and path.suffix.lower() in [".xlsx", ".zip", ".pdf"]:
            actual = digest(path)
            for name, expected in HASH.items():
                if actual == expected:
                    if name in found:
                        raise ValueError(f"Two identical inputs for {name} in data/raw")
                    found[name] = path
    required = {"RICH", "NAGALA", "NHANES"}
    if not required <= set(found):
        raise ValueError(f"Missing verified bytes for {sorted(required - set(found))}. See README.")
    return found


def safe_unpack(archive, dest):
    dest = Path(dest).resolve()
    with zipfile.ZipFile(archive) as z:
        seen = set()
        for item in z.infolist():
            n = PurePosixPath(item.filename.replace(chr(92), "/"))
            target = (dest / str(n)).resolve()
            if (
                not target.is_relative_to(dest)
                or ".." in n.parts
                or n.is_absolute()
                or (":" in str(n))
                or stat.S_ISLNK(item.external_attr >> 16)
                or (str(n).casefold() in seen)
            ):
                raise ValueError(f"Unsafe or duplicate ZIP member: {n}")
            seen.add(str(n).casefold())
        for item in z.infolist():
            if not item.is_dir():
                target = dest / item.filename.replace(chr(92), "/")
                with z.open(item) as src:

                    def copy(p):
                        with p.open("wb") as out:
                            shutil.copyfileobj(src, out)

                    atomic(target, copy)
        return [i.filename for i in z.infolist()]


def mapped(raw, mapping):
    if not raw.columns.is_unique or len(set(mapping.values())) != len(mapping):
        raise ValueError("Ambiguous columns")
    missing = set(mapping.values()) - set(raw)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    return raw[list(mapping.values())].rename(columns={v: k for k, v in mapping.items()}).copy()


def allowed(names, task):
    whitelist = PLUS if task == "A" else NONLAB
    if not names or len(names) != len(set(names)) or set(names) - set(whitelist):
        raise ValueError(f"Prohibited or duplicate predictor in {names}")
    return list(names)


def snapshot(df, cohort, idcol=None):
    return dict(
        cohort=cohort,
        rows=len(df),
        columns=len(df.columns),
        unique_id=df[idcol].nunique() if idcol else None,
        missing_id=df[idcol].isna().sum() if idcol else None,
        duplicate_id_rows=int(df[idcol].duplicated().sum()) if idcol else None,
        exact_duplicate_rows=int(df.duplicated().sum()),
        id_kind="source" if idcol else "no retained ID",
    )


def describe(df, cohort, units=None):
    out = []
    for name, col in df.items():
        num = pd.to_numeric(col, errors="coerce")
        levels = col.value_counts(dropna=True)
        qq = num.quantile([0, 0.01, 0.25, 0.5, 0.75, 0.99, 1]).tolist()
        out.append(
            dict(
                cohort=cohort,
                field=name,
                dtype=str(col.dtype),
                unit=(units or {}).get(name, "see source header / codebook"),
                n=len(col),
                missing=int(col.isna().sum()),
                missing_pct=100 * col.isna().mean(),
                zeros=int(num.eq(0).sum()),
                distinct=col.nunique(),
                levels=json.dumps({str(k): int(v) for k, v in levels.items()}) if len(levels) <= 35 else "",
                **dict(zip(["min", "p01", "q1", "median", "q3", "p99", "max"], qq)),
            )
        )
    return out


def workbook(path, cohort):
    sheet = {"RICH": "RC", "NAGALA": "Sheet 1"}[cohort]
    cached = openpyxl.load_workbook(path, read_only=True, data_only=True)
    formulas = openpyxl.load_workbook(path, read_only=True, data_only=False)
    inventory = []
    counts = Counter()
    result = None
    for s in cached:
        it = s.values
        headers = list(next(it))
        if len(headers) != len(set(headers)) or None in headers:
            raise ValueError(f"Invalid headers in {s.title}")
        raw = pd.DataFrame(it, columns=headers)
        inventory.append(
            dict(cohort=cohort, sheet=s.title, rows=len(raw), columns=len(headers), headers=headers)
        )
        for row in formulas[s.title].iter_rows(min_row=2):
            for c in row:
                if c.data_type == "f":
                    name = headers[c.column - 1]
                    val = raw.iloc[c.row - 2, c.column - 1]
                    counts[s.title, name, "formulas"] += 1
                    counts[s.title, name, "cache_empty"] += int(pd.isna(val))
                    counts[s.title, name, "cache_zero"] += int(val == 0)
        if s.title == sheet:
            result = raw
    cached.close()
    formulas.close()
    if result is None:
        raise ValueError(f"Missing sheet {sheet}")
    return (
        result,
        inventory,
        [dict(cohort=cohort, sheet=s, field=f, check=c, count=n) for (s, f, c), n in counts.items()],
    )


def harmonize(raw, cohort, keep_alt_zero=False):
    j = cohort == "NAGALA"
    d = mapped(raw, J if j else R)
    checks = []
    if j:
        for k, (source, factor) in CONVERT.items():
            reference = raw[source] * factor
            bad = (d[k].isna() | d[k].eq(0)) & reference.gt(0)
            good = d[k].notna() & ~bad & reference.notna()
            err = (d.loc[good, k] - reference[good]).abs()
            if (err > 0.002).any():
                raise ValueError(f"{k} mmol/L cache disagrees with mass concentration; inspect factors")
            checks.append(
                dict(
                    cohort=cohort,
                    field=k,
                    recovered=int(bad.sum()),
                    verified=int(good.sum()),
                    max_abs_error=err.max(),
                    factor=factor,
                )
            )
            d.loc[bad, k] = reference[bad]
        d["time"] = d.time / 365.25
        d["site"] = "NAGALA"
    for k in PLUS:
        if k not in ("sex", "smoking"):
            valid = d[k].ge(0) if k == "alt" and keep_alt_zero else d[k].gt(0)
            d[k] = d[k].where(valid & np.isfinite(d[k]))
    sexmap = {0: 0, 1: 1} if j else {1: 1, 2: 0}
    if set(d.sex.dropna().unique()) - set(sexmap):
        raise ValueError("Undocumented sex codes")
    d["sex"] = d.sex.map(sexmap)
    d["smoking"] = d.smoking.map({1: 0, 2: 0, 3: 1} if j else {1: 1, 2: 0, 3: 0})
    if d.id.isna().any() or d.id.duplicated().any():
        raise ValueError("Missing/repeated source ID; resolve person-level records")
    d["id"] = cohort + "-" + d.id.astype(str)
    d["cohort"] = cohort
    return (d, checks)


def status(d):
    return np.where((d.event == 1) & (d.time <= 3), 1, np.where(d.time > 3, 0, -1))


def screen_candidates(raw, period, no_code):
    d = mapped(raw, NH)
    d["id"] = period + "-row-" + pd.Series(np.arange(len(d)), index=d.index).astype(str)
    d["cohort"] = period
    d["site"] = period
    d["event"] = d.a1c.ge(6.5).astype(int).where(d.a1c.gt(0) & d.a1c.le(100))
    for k in ["age", "bmi", "sbp"]:
        d[k] = d[k].where(d[k].gt(0))
    if not set(d.sex.dropna()).issubset({0, 1}):
        raise ValueError("Unexpected NHANES sex codes")
    rules = [
        ("age >=20", d.age >= 20),
        ("valid measured HbA1c percent", d.event.notna()),
        (f"extract Diabetes_self equals {no_code}", d.reported.eq(no_code)),
    ]
    return flow(d, rules, period)


def flow(d, rules, cohort):
    rows = []
    stay = pd.Series(True, index=d.index)
    for label, mask in [("raw", stay.copy())] + rules:
        stay &= mask.fillna(False)
        subset = d.loc[stay]
        rows.append(
            dict(
                cohort=cohort,
                step=label,
                n=len(subset),
                events=int(subset.event.eq(1).sum()),
                cases3=int((status(subset) == 1).sum()) if "time" in d else None,
            )
        )
    return (d.loc[stay].reset_index(drop=True), rows)


def eligible(d, broad=False):
    rules = [
        ("age >=20", d.age >= 20),
        ("positive finite follow-up", d.time.gt(0) & np.isfinite(d.time)),
        ("binary event", d.event.isin([0, 1])),
        ("observed FPG <7 and source baseline exclusions", d.fpg.lt(7)),
    ]
    if "a1c" in d:
        rules.append(("observed baseline HbA1c >0 and <6.5%", d.a1c.gt(0) & d.a1c.lt(6.5)))
    if not broad:
        rules.append(("harmonized FPG <6.1", d.fpg.lt(6.1)))
    return flow(d, rules, d.cohort.iloc[0] + ("_broad" if broad else ""))


def audit(paths, out, cache, no_code=0):
    out, cache = (Path(out), Path(cache))
    rawdesc, summary, inventory, formulas, conversions, flows, baseline, sites, aux = (
        [],
        [],
        [],
        [],
        [],
        [],
        [],
        [],
        [],
    )
    by = {}
    availability = []
    for c, mp, shape in [("RICH", R, (211833, 25)), ("NAGALA", J, (15464, 31))]:
        raw, inv, fmt = workbook(paths[c], c)
        summary.append({**snapshot(raw, c, mp["id"]), "shape_matches_reference": raw.shape == shape})
        inventory += inv
        formulas += fmt
        rawdesc += describe(raw, c)
        d, chk = harmonize(raw, c)
        conversions += chk
        if c == "RICH":
            f = raw["FPG of final visit(mmol/L)"].ge(7)
            a = raw["Diabetes diagnosed during followup（1,Yes）"].fillna(0).eq(1)
            aux = (
                pd.DataFrame({"event": d.event, "final_FPG_ge7": f, "aux_reported": a})
                .value_counts()
                .rename("n")
                .reset_index()
                .to_dict("records")
            )
        for label, dd in [(c, d)] + ([("RICH_alt0", harmonize(raw, c, True)[0])] if c == "RICH" else []):
            for broad in [False, True] if label == "RICH" else [False]:
                name = label + ("_broad" if broad else "")
                cohort, log = eligible(dd, broad)
                for x in log:
                    x["cohort"] = name
                flows += log
                dump(cache / (name + ".pkl"), cohort)
                if name != label or name != c:
                    continue
                baseline += describe(cohort[PLUS + ["time", "event"]], c, UNITS)
                for setname, features in [("nonlaboratory", NONLAB), ("primary", CORE), ("expanded", PLUS)]:
                    availability.append(
                        dict(
                            cohort=c,
                            set=setname,
                            features=len(features),
                            n=len(cohort),
                            all_observed=int(cohort[features].notna().all(axis=1).sum()),
                            at_least_one_missing=int(cohort[features].isna().any(axis=1).sum()),
                        )
                    )
                for stage, part in [("raw", dd), ("eligible", cohort)]:
                    for site, g in list(part.groupby("site", dropna=False)) + [("ALL", part)]:
                        st = status(g)
                        sites.append(
                            dict(
                                cohort=c,
                                stage=stage,
                                site=site,
                                n=len(g),
                                events=int(g.event.eq(1).sum()),
                                cases3=int((st == 1).sum()),
                                controls3=int((st == 0).sum()),
                                early_or_at3=int((st == -1).sum()),
                                risk_set_3=int(g.time.ge(3).sum()),
                                followup_median=g.time.median(),
                            )
                        )
        dump(cache / (c + "_valid_masks.pkl"), d[["id"]].join(d[PLUS].notna().add_suffix("_observed")))
    members = safe_unpack(paths["NHANES"], cache / "nhanes_members")
    base = cache / "nhanes_members"
    for period, rows in zip(PERIODS, [8702, 6185]):
        files = [n for n in members if period in n and n.endswith(".csv")]
        prov = [n for n in members if period in n and n.endswith(".json")]
        if len(files) != 1 or len(prov) != 1:
            raise ValueError("Missing/ambiguous period CSV or provenance")
        raw = pd.read_csv(base / files[0])
        if list(raw) != NHC:
            raise ValueError(f"Unexpected {period} headers")
        meta = json.loads((base / prov[0]).read_text(encoding="utf-8-sig"))
        if meta["columns"]["HbA1c"] != "LBXGH" or meta["columns"]["Diabetes_self"] != "DIQ010":
            raise ValueError("Check endpoint provenance")
        by[period] = meta
        summary.append({**snapshot(raw, period), "shape_matches_reference": raw.shape == (rows, 21)})
        rawdesc += describe(raw, period)
        d, lg = screen_candidates(raw, period, no_code)
        for x in lg:
            x["status"] = "candidate count pending source recode review"
        flows += lg
        dump(cache / (period + ".pkl"), d)
        baseline += describe(d[NONLAB + ["a1c", "event"]], period, UNITS)
    reports = [n for n in members if n.endswith(".md")]
    if len(reports) != 1:
        raise ValueError("Missing consistency report")
    js(
        out / "inventory.json",
        dict(
            hashes={n: digest(p) for n, p in paths.items()},
            workbooks=inventory,
            zip_members=members,
            provenance=by,
            consistency_report=(base / reports[0]).read_text(encoding="utf-8-sig"),
        ),
    )
    for name, rows in dict(
        structure=summary,
        raw_features=rawdesc,
        formula_cache=formulas,
        unit_checks=conversions,
        cohort_flow=flows,
        baseline=baseline,
        site_support=sites,
        feature_set_availability=availability,
        rich_outcome_check=aux,
    ).items():
        csv(out / (name + ".csv"), rows)
    from dictionary import dictionary

    csv(out / "field_dictionary.csv", dictionary(pd.DataFrame(rawdesc)))
    return pd.DataFrame(summary)

## 03 · Package module `dictionary.py`: field dictionary

Builds a field dictionary from the raw-feature audit: type, unit, missingness and range of every source column.

In [ ]:
%%writefile ckinn_obj1/dictionary.py
import re
import pandas as pd
from data import R, J, NH, CORE, PLUS, UNITS


def dictionary(raw_profile):
    rows = []
    nh_units = dict(
        Age="years",
        BMI="kg/m²",
        SystolicBP="mmHg",
        DiastolicBP="mmHg",
        HbA1c="%",
        FastingGlucose="mg/dL",
        MentHlth="derived PHQ-9 coding; see provenance and observed levels",
        Income="derived category",
        Education="derived category",
    )
    for row in raw_profile.to_dict("records"):
        c = row["cohort"]
        f = row["field"]
        mp = R if c == "RICH" else J if c == "NAGALA" else NH
        alias = next((k for k, v in mp.items() if v == f), "")
        unit = UNITS.get(alias, "")
        role = "audit only"
        code = "Numeric measurement; unusual positive values retained."
        if alias in CORE:
            role = "shared primary" if c in ["RICH", "NAGALA"] else "screening primary"
        elif alias in PLUS:
            role = "expanded sensitivity"
        elif alias in ("event", "time", "id", "site", "a1c", "reported"):
            role = {
                "event": "outcome",
                "time": "follow-up",
                "id": "partition key",
                "site": "group only",
                "a1c": "baseline eligibility / screening label",
                "reported": "screening eligibility",
            }[alias]
        if c in ("RICH", "NAGALA"):
            if alias == "sex":
                code = "1 male, 2 female" if c == "RICH" else "1 male, 0 female; header conflicts"
            elif alias == "smoking":
                code = "1 current, 2 ever, 3 never" if c == "RICH" else "1 never, 2 past, 3 current"
            elif alias == "event":
                code = "1 incident detection, 0 censored"
            elif alias == "id":
                code = "Nonmissing unique source participant ID"
            elif alias == "site":
                code = "Observed grouping codes; center mapping unavailable"
            elif alias == "time":
                unit = "days" if c == "NAGALA" else "years"
                code = "Positive observed duration"
            elif alias == "a1c":
                code = "Baseline percent; eligible 0 < HbA1c < 6.5"
            elif alias in PLUS:
                code = (
                    "Finite and positive; ALT zero retained only in sensitivity"
                    if alias == "alt"
                    else "Finite and positive"
                )
            elif any(
                (
                    s in f.lower()
                    for s in (
                        "status",
                        "0/1",
                        "consumption, non:",
                        "phenotype",
                        "histroy",
                        "diagnosed during",
                    )
                )
            ):
                code = "Source header coding; audit only"
            if not unit:
                known = [
                    ("mmol/mol", "mmol/mol"),
                    ("mmol/l", "mmol/L"),
                    ("umol/l", "µmol/L"),
                    ("mg/dl", "mg/dL"),
                    ("iu/l", "U/L"),
                    ("(cm)", "cm"),
                    (", cm", "cm"),
                    ("(kg)", "kg"),
                    (", kg", "kg"),
                    ("g/wk", "g/week"),
                ]
                unit = next((u for s, u in known if s in f.lower()), "source ID / code")
        else:
            unit = nh_units.get(f, "derived category / binary code")
            code = "Producer transformation unverified; see observed levels and JSON source variable."
            if f == "Sex":
                code = "0 and 1; category labels unverified"
            if f == "Diabetes_self":
                code = "Extract 0/1; source DIQ010 uses 1/2/3/7/9; as-coded eligibility uses 0"
            if f == "HbA1c":
                code = "LBXGH percent >0 and ≤100; single screen positive ≥6.5%"
            if f in ("BMI", "SystolicBP", "Age"):
                code = "Finite and positive; eligibility age ≥20"
            if f in ("HbA1c", "FastingGlucose", "Diabetes_lab", "Diabetes_self"):
                role += "; prohibited predictor"
        rows.append(
            dict(
                cohort=c,
                raw_field=f,
                canonical=alias,
                unit=unit,
                role=role,
                validity_or_codes=code,
                observed_levels=row["levels"],
            )
        )
    return pd.DataFrame(rows)

## 04 · Package module `stats.py`: fixed-horizon estimands

Reverse Kaplan–Meier censoring survival (detections precede withdrawals at tied times), IPCW weights, the IPCW Brier score over all N (Graf et al., 1999), the weighted cumulative/dynamic AUROC, average precision, calibration intercept (logit offset) and slope, decile calibration, threshold metrics and the paired participant bootstrap with censoring weights re-estimated in every draw.

In [ ]:
%%writefile ckinn_obj1/stats.py
import numpy as np
import pandas as pd
from scipy.special import expit, logit
from scipy.optimize import minimize, brentq
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from data import status


class ReverseKM:

    def __init__(self, t, e):
        t = np.asarray(t, float)
        e = np.asarray(e)
        if len(t) == 0 or not np.isfinite(t).all() or (t <= 0).any() or (not np.isin(e, [0, 1]).all()):
            raise ValueError("Invalid survival data")
        self.times, idx, n = np.unique(t, return_inverse=True, return_counts=True)
        deaths = np.bincount(idx, weights=e, minlength=len(n))
        risk = len(t) - np.r_[0, np.cumsum(n)[:-1]]
        fraction = np.divide(n - deaths, risk - deaths, out=np.zeros(len(n)), where=risk > deaths)
        self.G = np.r_[1.0, np.cumprod(1.0 - fraction)]

    def at(self, q, left=False):
        return self.G[np.searchsorted(self.times, q, side="left" if left else "right")]


def weights(d, by_site=False):
    if "time" not in d:
        return (d.event.to_numpy(float), np.ones(len(d)), np.nan)
    st = status(d)
    y = (st == 1).astype(float)
    w = np.zeros(len(d))
    km = ReverseKM(d.time, d.event)
    groups = [np.arange(len(d))]
    if by_site:
        sites = d.site.fillna("missing").astype(str)
        tails = (d.time > 3).groupby(sites).sum()
        if tails.max() < 30:
            raise ValueError("No site with 30 observed beyond three years")
        pooled = sites.where(sites.map(tails) >= 30, tails.idxmax())
        groups = d.groupby(pooled).indices.values()
    for inds in groups:
        part = d.iloc[inds]
        g = ReverseKM(part.time, part.event) if by_site else km
        if g.at(3) <= 0 or not (part.time > 3).any():
            raise ValueError("No three-year censoring support")
        cases = inds[st[inds] == 1]
        w[cases] = 1 / g.at(d.time.to_numpy()[cases], left=True)
        w[inds[st[inds] == 0]] = 1 / g.at(3)
    return (y, w, float(km.at(3)))


def brier(y, w, p):
    return float(np.mean(w * np.square(y - np.asarray(p, float))))


def support(d):
    y, w, g = weights(d)
    known = w > 0
    return dict(
        n=len(y),
        events=int((y == 1).sum()),
        controls=int(sum(known & (y == 0))),
        unknown=int(sum(~known)),
        G3=g,
        at_risk_3=int((d.time >= 3).sum()) if "time" in d else None,
        ESS=w.sum() ** 2 / (w @ w),
        positive_w_min=w[known].min(),
        positive_w_median=np.median(w[known]),
        positive_w_p99=np.quantile(w[known], 0.99),
        w_max=w.max(),
    )


def point_threshold(y, w, p):
    c = (y == 0) & (w > 0)
    if not c.any():
        raise ValueError("No known controls to choose threshold")
    v = np.argsort(p[c], kind="stable")
    k = np.searchsorted(np.cumsum(w[c][v]), 0.9 * w[c].sum())
    return float(np.nextafter(float(p[c][v[k]]), np.inf))


def calibration(y, w, p):
    l = logit(np.clip(p, 1e-07, 1 - 1e-07))
    if np.unique(y[w > 0]).size < 2:
        return (np.nan, np.nan)
    intercept = brentq(lambda a: np.sum(w * (expit(a + l) - y)), -50, 50)
    if np.ptp(l[w > 0]) < 1e-10:
        return (intercept, np.nan)
    X = np.column_stack([np.ones(len(y)), l])

    def loss(beta):
        eta = X @ beta
        return (np.average(np.logaddexp(0, eta) - y * eta, weights=w), X.T @ (w * (expit(eta) - y)) / w.sum())

    opt = minimize(loss, [0, 1], jac=True, method="BFGS", options={"gtol": 1e-08})
    return (intercept, opt.x[1] if opt.success and abs(opt.x[1]) < 100 else np.nan)


def calbins(y, w, p):
    p = np.asarray(p)
    labels = np.searchsorted(np.unique(np.quantile(p, np.linspace(0, 1, 11)[1:-1])), p, side="left")
    rows = []
    for k in np.unique(labels):
        pick = labels == k
        rows.append(
            dict(
                bin=int(k),
                n=int(pick.sum()),
                predicted=p[pick].mean(),
                observed=np.average(y[pick], weights=w[pick]) if w[pick].sum() > 0 else np.nan,
            )
        )
    return pd.DataFrame(rows)


def metrics(d, p, threshold, by_site=False):
    p = np.asarray(p, dtype=float)
    if len(p) != len(d) or not np.isfinite(p).all() or np.any((p < 0) | (p > 1)):
        raise ValueError("Invalid probabilities")
    y, w, g = weights(d, by_site)
    k = w > 0
    s = p >= threshold
    tn, fp, fn, tp = [w[(y == a) & (s == b)].sum() for a, b in [(0, 0), (0, 1), (1, 0), (1, 1)]]

    def ratio(a, b):
        return a / b if b else np.nan

    two = np.unique(y[k]).size == 2
    ci, cs = calibration(y, w, p)
    cb = calbins(y, w, p)
    return dict(
        **support(d),
        brier=brier(y, w, p),
        auroc=roc_auc_score(y[k], p[k], sample_weight=w[k]) if two else np.nan,
        average_precision=average_precision_score(y[k], p[k], sample_weight=w[k]) if two else np.nan,
        predicted=p.mean(),
        observed=np.mean(w * y),
        cal_intercept=ci,
        cal_slope=cs,
        ece=np.average(abs(cb.predicted - cb.observed), weights=cb.n),
        threshold=threshold if np.ndim(threshold) == 0 else "fold-specific",
        sensitivity=ratio(tp, tp + fn),
        specificity=ratio(tn, tn + fp),
        ppv=ratio(tp, tp + fp),
        npv=ratio(tn, tn + fn),
        f1=ratio(2 * tp, 2 * tp + fp + fn),
        fpr=ratio(fp, fp + tn),
        fnr=ratio(fn, fn + tp)
    )


def bootstrap(d, p1, p0, B, seed, threshold=None, all_metrics=False):
    rng = np.random.default_rng(seed)
    vals = []
    failures = 0
    for _ in range(B):
        i = rng.integers(len(d), size=len(d))
        dd = d.iloc[i].reset_index(drop=True)
        try:
            y, w, _ = weights(dd)
            if np.unique(y[w > 0]).size < 2:
                raise ValueError("single class")
            if all_metrics:
                vals.append(metrics(dd, p1[i], threshold if np.ndim(threshold) == 0 else threshold[i]))
            else:
                vals.append(dict(delta=brier(y, w, p1[i]) - brier(y, w, p0[i])))
        except (ValueError, FloatingPointError):
            failures += 1
    frame = pd.DataFrame(vals)
    result = {}
    for k in (
        [
            "brier",
            "auroc",
            "average_precision",
            "cal_intercept",
            "cal_slope",
            "ece",
            "predicted",
            "observed",
            "sensitivity",
            "specificity",
            "fpr",
            "fnr",
            "ppv",
            "npv",
            "f1",
        ]
        if all_metrics
        else ["delta"]
    ):
        v = frame[k].dropna() if k in frame else []
        lo, hi = np.quantile(v, [0.025, 0.975]) if len(v) >= max(2, 0.8 * B) else (np.nan, np.nan)
        result.update({k + "_lo": lo, k + "_hi": hi, k + "_valid_boot": len(v)})
    return dict(**result, requested_boot=B, failed_boot=failures)

## 05 · Package module `models.py`: preprocessing, networks, penalty and comparators

Preprocessing is fitted on training rows only: median imputation, standardisation and one missingness indicator per input. The branched network has demographic, body/pressure and laboratory branches feeding a fusion block that outputs one logit. The clinical penalty averages the squared wrong-signed probability derivative over observed, in-support participant–feature pairs, and is differentiated during training. The module also covers training with early stopping and epoch checkpoints, participant-disjoint stratified splits, IPCW-weighted comparators, the SVM with cross-fitted Platt calibration, and ridge Cox truncated at three years.

In [ ]:
%%writefile ckinn_obj1/models.py
from pathlib import Path
import copy, random, inspect, time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold
from scipy.special import expit
from lifelines import CoxPHFitter
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from data import CORE, PLUS, NONLAB, allowed, dump, restore, key, status
from stats import weights, brier

DIRECTIONS = dict(fpg=1, bmi=1, age=1, sbp=1, tg=1, hdl=-1)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.set_num_threads(1)


class Prep:

    def __init__(self, names, task, masks=True):
        self.names = allowed(names, task)
        self.task = task
        self.masks = masks

    def values(self, d):
        x = d.loc[:, self.names].to_numpy(dtype=float, copy=True)
        x[~np.isfinite(x)] = np.nan
        return x

    def fit(self, d):
        x = self.values(d)
        if np.all(np.isnan(x), axis=0).any():
            raise ValueError("Training partition has an entirely missing feature")
        self.med = np.nanmedian(x, axis=0)
        filled = np.where(np.isnan(x), self.med, x)
        self.mu = filled.mean(0)
        self.scale = filled.std(0)
        self.scale[self.scale < 1e-12] = 1.0
        self.lo, self.hi = np.nanquantile(x, [0.01, 0.99], axis=0)
        return self

    def transform(self, d):
        x = self.values(d)
        obs = ~np.isnan(x)
        z = (np.where(obs, x, self.med) - self.mu) / self.scale
        features = np.column_stack([z, (~obs).astype(float)]) if self.masks else z
        return (features.astype("float32"), obs & (x >= self.lo) & (x <= self.hi))


class Net(nn.Module):

    def __init__(self, prep, h):
        super().__init__()
        names = prep.names
        width = h["branch"]
        self.groups = []
        width_in = len(names) * (1 + int(prep.masks))
        if h.get("topology", "branches") == "branches":
            for group in (
                ["age", "sex", "smoking"],
                ["bmi", "sbp", "dbp"],
                ["alt", "fpg", "tg", "hdl", "tc", "ast"],
            ):
                inds = [names.index(k) for k in group if k in names]
                self.groups.append(inds + [i + len(names) for i in inds] if prep.masks else inds)
            self.branches = nn.ModuleList(
                [nn.Sequential(nn.Linear(max(len(ix), 1), width), nn.SiLU()) for ix in self.groups]
            )
            width_in = 3 * width
        h1, h2 = h["fusion"]
        self.fusion = nn.Sequential(
            nn.Linear(width_in, h1),
            nn.SiLU(),
            nn.Dropout(h["dropout"]),
            nn.Linear(h1, h2),
            nn.SiLU(),
            nn.Linear(h2, 1),
        )

    def forward(self, x):
        if self.groups:
            x = torch.cat(
                [
                    b(x[:, ix] if ix else x.new_zeros((len(x), 1)))
                    for ix, b in zip(self.groups, self.branches)
                ],
                dim=1,
            )
        return self.fusion(x).flatten()


def clinical(p, x, inside, names, directions):
    if not directions:
        return x.new_zeros(())
    idx = [names.index(k) for k in directions]
    signs = x.new_tensor([DIRECTIONS[k] for k in directions])
    g = torch.autograd.grad(p.sum(), x, create_graph=True)[0][:, idx]
    terms = torch.relu(-g * signs).square()
    ok = inside[:, idx]
    return (terms * ok).sum() / ok.sum().clamp_min(1)


def side_weights(d):
    sites = d.site.fillna("missing").astype(str)
    counts = sites.value_counts()
    pooled = sites.where(sites.map(counts) >= 50, "__sparse__")
    counts = pooled.value_counts()
    raw = np.minimum(3.0, len(d) / (len(counts) * pooled.map(counts).to_numpy(float)))
    return raw / raw.mean()


def loss_terms(logits, x, inside, y, w, sex, prep, h):
    loss = w * F.binary_cross_entropy_with_logits(logits, y, reduction="none")
    total = loss.mean()
    if h["lambda"]:
        total = total + h["lambda"] * clinical(logits.sigmoid(), x, inside, prep.names, h["directions"])
    if h.get("group_penalty", 0):
        m = sex == 1
        f = sex == 0
        if m.any() and f.any():
            total = total + h["group_penalty"] * (loss[m].mean() - loss[f].mean()).square()
    return total


def torch_risk(net, x, device="cpu", batch=2048):
    net.eval()
    with torch.no_grad():
        return np.concatenate(
            [
                net(torch.from_numpy(z).to(device)).sigmoid().cpu().numpy()
                for z in np.array_split(x, max(1, int(np.ceil(len(x) / batch))))
            ]
        ).astype(float)


def train_net(d, valid, prep, h, seed, epochs, patience, device, checkpoint, identity, interrupt_epoch=None):
    seed_all(seed)
    net = Net(prep, h).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=h["lr"], weight_decay=h["decay"])
    X, inside = prep.transform(d)
    y, w, _ = weights(d, h.get("site_g", False))
    if h.get("site_balance"):
        w *= side_weights(d)
    if valid is not None:
        xv = prep.transform(valid)[0]
        vy, vw, _ = weights(valid, h.get("site_g", False))
    state = dict(epoch=0, best=np.inf, best_epoch=0, best_state=None, wait=0)
    generator = np.random.default_rng(seed)
    if checkpoint and Path(checkpoint).exists():
        saved = restore(checkpoint)
        if saved["identity"] != identity:
            raise ValueError("Incompatible epoch checkpoint")
        state = saved["progress"]
        net.load_state_dict(saved["model"])
        opt.load_state_dict(saved["optimizer"])
        random.setstate(saved["python_rng"])
        np.random.set_state(saved["numpy_rng"])
        generator.bit_generator.state = saved["shuffle_rng"]
        torch.set_rng_state(saved["torch_rng"])
        if device == "cuda":
            torch.cuda.set_rng_state_all(saved["cuda_rng"])
    for epoch in range(state["epoch"], epochs):
        if valid is not None and state["wait"] >= patience:
            break
        net.train()
        order = generator.permutation(len(d))
        for start in range(0, len(d), h["batch"]):
            i = order[start : start + h["batch"]]
            x = torch.from_numpy(X[i]).to(device).requires_grad_(bool(h["lambda"]))

            def tensor(a):
                return torch.as_tensor(a, device=device, dtype=torch.float32)

            logits = net(x)
            loss = loss_terms(
                logits,
                x,
                torch.from_numpy(inside[i]).to(device),
                tensor(y[i]),
                tensor(w[i]),
                tensor(d.sex.to_numpy()[i]),
                prep,
                h,
            )
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite neural loss")
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        state["epoch"] = epoch + 1
        if valid is not None:
            score = brier(vy, vw, torch_risk(net, xv, device))
            if score < state["best"] - 1e-10:
                best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
                state.update(best=score, best_epoch=epoch + 1, best_state=best_state, wait=0)
            else:
                state["wait"] += 1
        if checkpoint:
            cpu_state = {k: v.detach().cpu() for k, v in net.state_dict().items()}
            optstate = copy.deepcopy(opt.state_dict())
            for slots in optstate["state"].values():
                for k, v in slots.items():
                    if torch.is_tensor(v):
                        slots[k] = v.cpu()
            dump(
                checkpoint,
                dict(
                    identity=identity,
                    preprocessing=prep,
                    model=cpu_state,
                    optimizer=optstate,
                    scheduler=None,
                    progress=state,
                    python_rng=random.getstate(),
                    numpy_rng=np.random.get_state(),
                    shuffle_rng=generator.bit_generator.state,
                    torch_rng=torch.get_rng_state(),
                    cuda_rng=torch.cuda.get_rng_state_all() if device == "cuda" else [],
                ),
            )
        if interrupt_epoch == epoch + 1:
            raise InterruptedError("synthetic interruption")
    if valid is not None:
        net.load_state_dict(state["best_state"])
    return (net.cpu().eval(), state["best_epoch"] if valid is not None else state["epoch"])


def split_indices(d, k, seed):
    if not d.id.is_unique:
        raise ValueError("Repeated person or row key")
    base = pd.Series(status(d) if "time" in d else d.event.to_numpy(), dtype=str)
    if base.value_counts().min() < k:
        raise ValueError(f"Too few participants in a status stratum for {k} folds")
    joint = base + "/" + d.site.reset_index(drop=True).astype(str)
    strata = joint.copy()
    for label in base.unique():
        block = base == label
        counts = joint[block].value_counts()
        rare = block & joint.isin(counts[counts < k].index)
        if rare.any():
            strata.loc[rare] = label + "/pooled" if rare.sum() >= k else counts.idxmax()
    return list(StratifiedKFold(k, shuffle=True, random_state=seed).split(np.zeros(len(d)), strata))


def fit_weighted(model, x, y, w):
    if "sample_weight" not in inspect.signature(model.fit).parameters:
        raise TypeError("Estimator lacks sample_weight")
    good = w > 0
    if np.unique(y[good]).size < 2:
        raise ValueError("Single known training class")
    model.fit(x[good], y[good], sample_weight=w[good])
    return model


def estimator(name, h, seed):
    if name == "LR":
        return LogisticRegression(C=h["C"], max_iter=500)
    if name == "RF":
        return RandomForestClassifier(
            n_estimators=h["trees"], max_depth=h["depth"], min_samples_leaf=8, n_jobs=1, random_state=seed
        )
    if name == "XGB":
        return XGBClassifier(
            n_estimators=h["trees"],
            max_depth=h["depth"],
            learning_rate=0.05,
            tree_method="hist",
            random_state=seed,
            n_jobs=1,
        )
    if name == "LGB":
        return LGBMClassifier(
            n_estimators=h["trees"],
            num_leaves=2 ** h["depth"],
            max_depth=h["depth"],
            learning_rate=0.05,
            verbosity=-1,
            n_jobs=1,
            random_state=seed,
        )
    if name == "SVM":
        return LinearSVC(C=h["C"], dual=False, max_iter=3000, random_state=seed)
    raise ValueError(name)


class Fitted:

    def __init__(self, kind, prep, obj, extra=None):
        self.kind, self.prep, self.obj, self.extra = (kind, prep, obj, extra)

    def predict(self, d):
        x = self.prep.transform(d)[0]
        if self.kind == "net":
            return torch_risk(self.obj, x)
        if self.kind == "reference":
            return np.full(len(x), self.obj)
        if self.kind == "Cox":
            pos, H = self.extra
            hazard = self.obj.predict_partial_hazard(
                pd.DataFrame(x[:, pos], columns=[f"x{i}" for i in pos])
            ).to_numpy()
            return -np.expm1(-H * hazard)
        if self.kind == "SVM":
            return self.extra.predict_proba(self.obj.decision_function(x).reshape(-1, 1))[:, 1]
        return self.obj.predict_proba(x)[:, 1]


def fit_model(kind, d, valid, h, task, seed, epochs, patience, device, checkpoint, identity):
    pp = Prep(h["features"], task, h.get("masks", True)).fit(d)
    if kind == "net":
        model, ep = train_net(d, valid, pp, h, seed, epochs, patience, device, checkpoint, identity)
        return (Fitted(kind, pp, model), ep)
    x = pp.transform(d)[0]
    y, w, _ = weights(d)
    if kind == "reference":
        return (Fitted(kind, pp, np.average(y, weights=w)), 0)
    if kind == "Cox":
        pos = np.flatnonzero(x.std(0) > 1e-08)
        cd = pd.DataFrame(x[:, pos], columns=[f"x{i}" for i in pos])
        cd["T"] = np.minimum(d.time, 3).to_numpy()
        cd["E"] = ((d.time <= 3) & (d.event == 1)).to_numpy()
        if not cd.E.any():
            raise ValueError("No training detections by three")
        model = CoxPHFitter(penalizer=h["ridge"], l1_ratio=0.0).fit(cd, duration_col="T", event_col="E")
        H = float(model.baseline_cumulative_hazard_.iloc[-1, 0])
        return (Fitted(kind, pp, model, (pos, H)), 0)
    calibrator = None
    if kind == "SVM":
        scores = np.empty(len(d))
        wy = np.empty(len(d))
        folds = split_indices(d, 2, seed)
        if checkpoint:
            sp = Path(checkpoint).with_name("svm_calibration_folds.pkl")
            if sp.exists():
                saved = restore(sp)
                if saved["ids"] != d.id.tolist():
                    raise ValueError("SVM calibration partition mismatch")
                folds = saved["folds"]
            else:
                dump(sp, dict(ids=d.id.tolist(), folds=folds))
        for a, b in folds:
            train, val = (d.iloc[a], d.iloc[b])
            p = Prep(h["features"], task, h.get("masks", True)).fit(train)
            yt, wt, _ = weights(train)
            m = fit_weighted(estimator(kind, h, seed), p.transform(train)[0], yt, wt)
            scores[b] = m.decision_function(p.transform(val)[0])
            wy[b] = weights(val)[1]
        calibrator = fit_weighted(LogisticRegression(C=10000.0, max_iter=500), scores[:, None], y, wy)
    model = fit_weighted(estimator(kind, h, seed), x, y, w)
    return (Fitted(kind, pp, model, calibrator), 0)

## 06 · Package module `architecture.py`: architecture figure

Draws the default eight-input architecture with its two training terms.

In [ ]:
%%writefile ckinn_obj1/architecture.py
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch


def draw(path):
    fig, ax = plt.subplots(figsize=(12, 7), layout="constrained")
    ax.set(xlim=(0, 12), ylim=(0, 7))
    ax.axis("off")
    ink = "#103341"

    def box(x, y, w, h, title, sub, fill):
        ax.add_patch(
            FancyBboxPatch(
                (x, y),
                w,
                h,
                boxstyle="round,pad=0.05,rounding_size=0.1",
                facecolor=fill,
                edgecolor="#71969a",
                linewidth=1,
            )
        )
        ax.text(
            x + w / 2,
            y + h * 0.68,
            title,
            ha="center",
            va="center",
            fontsize=12,
            fontweight="bold",
            color=ink,
        )
        ax.text(x + w / 2, y + h * 0.29, sub, ha="center", va="center", fontsize=10, color=ink)

    def line(a, b):
        ax.add_patch(FancyArrowPatch(a, b, arrowstyle="-|>", color="#4a7277", mutation_scale=14, lw=1.2))

    ax.text(0.25, 6.72, "Predicting recorded diabetes at three years", fontsize=21, weight="bold", color=ink)
    ax.text(
        0.25,
        6.3,
        "Baseline inputs use fold-fitted medians, positive scales and missingness indicators.",
        fontsize=12,
        color=ink,
    )
    for x, title, sub, n in [
        (0.25, "Demographics", "Age · sex", 4),
        (4.25, "Body and pressure", "BMI · systolic BP", 4),
        (8.25, "Laboratory", "ALT · FPG · TG · HDL-C", 8),
    ]:
        box(x, 4.96, 3.48, 0.95, title, sub, "#e6f2f0")
        ax.text(x + 1.74, 4.71, f"{n} inputs → Linear → SiLU → 8 values", ha="center", fontsize=10, color=ink)
        line((x + 1.74, 4.54), (6, 4.02))
    box(
        2.12,
        3.05,
        7.76,
        0.94,
        "Concatenate 24 learned values",
        "Linear 24 → 32 · SiLU · dropout · Linear 32 → 16 · SiLU",
        "#edf2f6",
    )
    line((6, 3.0), (6, 2.67))
    box(3.85, 1.76, 4.3, 0.87, "Linear 16 → 1 logit", "Sigmoid gives a three-year probability", "#cfe9e2")
    line((4.7, 1.71), (3.0, 1.21))
    line((7.3, 1.71), (9.0, 1.21))
    box(
        0.52,
        0.15,
        5.05,
        1.03,
        "IPCW outcome loss",
        "Weighted BCE with logits, averaged over all rows",
        "#f4f2e9",
    )
    box(
        6.42,
        0.15,
        5.05,
        1.03,
        "Soft clinical penalty",
        "Wrong-way probability gradients for observed FPG and BMI",
        "#f4f2e9",
    )
    ax.text(11.75, 6.72, "DEFAULT  ·  1,497 PARAMETERS", ha="right", fontsize=9, color="#487075")
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=155, bbox_inches="tight", facecolor="white")
    plt.close(fig)

## 07 · Package module `runner.py`: nested validation, selection, freezing and scoring

`Study` binds the run identity and directories. Its main methods are:

* `internal`: nested cross-validation in RICH, plus the held-out-site, broader-eligibility and ALT-zero variants;
* `plan`: inner-fold tuning, including the matched-penalty and component variants;
* `freeze`: final RICH fits and the freeze manifest;
* `score`: frozen scoring of the external cohort.

Candidate schedules for M1 and M3 share seeds, batch order and stopping rules. M3 additionally searches λ.

In [ ]:
%%writefile ckinn_obj1/runner.py
from pathlib import Path
from importlib.metadata import version
import json, platform, time, sys, traceback
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from data import *
from stats import weights, metrics, brier, point_threshold, support
from models import fit_model, split_indices


def env_info():
    from importlib.metadata import PackageNotFoundError

    versions = {}
    for name in [
        "numpy",
        "pandas",
        "scipy",
        "scikit-learn",
        "torch",
        "lifelines",
        "lightgbm",
        "xgboost",
        "openpyxl",
        "matplotlib",
        "nbformat",
        "shap",
    ]:
        try:
            versions[name] = version(name)
        except PackageNotFoundError:
            versions[name] = "absent"
    return dict(
        python=sys.version,
        platform=platform.platform(),
        packages=versions,
        cuda=torch.version.cuda,
        device_name=torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU",
    )


def model_spec(model, base, task):
    h = dict(base)
    h.setdefault("features", CORE if task == "A" else NONLAB)
    h.setdefault("directions", ["fpg", "bmi"] if task == "A" else ["bmi"])
    if model in ("M1", "fixed_zero", "nonlab_M1"):
        h["lambda"] = 0.0
    changes = {
        "M4": dict(site_balance=True),
        "M5": dict(site_balance=True, group_penalty=0.25),
        "fpg_only": dict(directions=["fpg"]),
        "bmi_only": dict(directions=["bmi"]),
        "expanded_prior": dict(directions=["fpg", "bmi", "age", "sbp", "tg", "hdl"]),
        "expanded_inputs": dict(features=PLUS),
        "no_masks": dict(masks=False),
        "single_stream": dict(topology="single"),
        "nonlab_M1": dict(features=NONLAB),
        "site_ipcw": dict(site_g=True),
    }
    h.update(changes.get(model, {}))
    return h


MATCHED_PAIR = ["fixed_zero", "fixed_positive"]
PRIOR_VARIANTS = ["fpg_only", "bmi_only", "expanded_prior"]
OTHER_VARIANTS = ["M4", "M5", "expanded_inputs", "no_masks", "single_stream", "nonlab_M1"]
VARIANTS = MATCHED_PAIR + PRIOR_VARIANTS + OTHER_VARIANTS
CLASSIC = ["reference", "LR", "RF", "XGB", "LGB", "SVM", "Cox"]


class Study:

    def __init__(self, root, cfg, paths=None):
        self.root = Path(root)
        self.cfg = cfg
        if cfg["tau"] != 3:
            raise ValueError("This protocol defines a three-year endpoint")
        self.mode = cfg["profile"]
        self.b = cfg["profiles"][self.mode]
        self.seed = cfg["seed"]
        self.device = (
            cfg["device"] if cfg["device"] != "auto" else "cuda" if torch.cuda.is_available() else "cpu"
        )
        self.paths = paths or discover(self.root / "data/raw")
        self.code = {p.name: digest(p) for p in sorted(Path(__file__).parent.glob("*.py"))}
        self.identity = dict(
            data={k: digest(v) for k, v in self.paths.items()},
            code=key(self.code),
            config=key({k: v for k, v in cfg.items() if k != "stage"}),
            device=self.device,
            runtime={k: v for k, v in env_info().items() if k != "packages"},
            versions=env_info()["packages"],
        )
        self.run = key(self.identity)[:16]
        cache_runtime = [platform.python_version(), pd.__version__, np.__version__, version("openpyxl")]
        self.cache = (
            self.root
            / "data/processed"
            / key(
                [
                    self.identity["data"],
                    self.code["data.py"],
                    self.code["dictionary.py"],
                    cfg["options"]["screen_no_code"],
                    cache_runtime,
                ]
            )[:16]
        )
        self.auditpath = self.root / "results/tables/audit" / self.cache.name
        self.private = self.root / "checkpoints" / self.mode / self.run
        self.table = self.root / "results/tables" / self.mode / self.run
        self.fig = self.root / "results/figures" / self.mode / self.run
        self.report = self.root / "results/reports" / self.mode / self.run
        self.log = self.root / "logs" / self.mode / self.run
        for p in [self.cache, self.private, self.table, self.fig, self.report, self.log]:
            p.mkdir(parents=True, exist_ok=True)
        js(self.root / "config" / (self.run + ".json"), cfg)
        js(self.log / "environment.json", env_info())
        self.mem = {}

    def audit(self):
        done = self.cache / "audit_complete.json"
        if not done.exists():
            report = audit(self.paths, self.auditpath, self.cache, self.cfg["options"]["screen_no_code"])
            if not report.shape_matches_reference.all():
                raise ValueError("Shape discrepancy. Inspect audit before proceeding.")
            js(done, {"signatures": {p.name: digest(p) for p in self.cache.glob("*.pkl")}})
        return pd.read_csv(self.auditpath / "structure.csv")

    def data(self, name):
        if name not in self.mem:
            if not (self.cache / "audit_complete.json").exists():
                raise RuntimeError("Run audit first")
            path = self.cache / (name + ".pkl")
            if (
                digest(path)
                != json.loads((self.cache / "audit_complete.json").read_text())["signatures"][path.name]
            ):
                raise ValueError("Validated cache changed")
            d = restore(path)
            cap = self.b["sample"]
            if cap and len(d) > cap:
                s = status(d) if "time" in d else d.event
                i, _ = train_test_split(np.arange(len(d)), train_size=cap, stratify=s, random_state=self.seed)
                d = d.iloc[np.sort(i)]
            self.mem[name] = d.reset_index(drop=True)
        return self.mem[name]

    def sources(self, task):
        if task == "B":
            opt = self.cfg["options"]
            if opt["screening"] == "verified" and (not opt["producer_recode_evidence"].strip()):
                raise ValueError(
                    "Verified screening requires the producer's documented crosswalk, including unknown/refused"
                )
            if opt["screening"] not in ("as_coded", "verified"):
                raise ValueError("Screening disabled")
            return PERIODS
        return ["RICH", "NAGALA"]

    def tag(self, task):
        return task if task == "A" else "B_" + self.cfg["options"]["screening"]

    def split(self, d, k, label, seed):
        path = self.private / "splits" / (label + ".pkl")
        ids = d.id.tolist()
        if path.exists():
            saved = restore(path)
            if saved["ids"] != ids or saved["k"] != k:
                raise ValueError("Immutable partition mismatch")
            return saved["folds"]
        folds = split_indices(d, k, seed)
        for a, b in folds:
            assert not set(d.id.iloc[a]).intersection(d.id.iloc[b])
        dump(path, dict(ids=ids, k=k, seed=seed, folds=folds))
        return folds

    def candidates(self, name, task):
        grid = self.cfg["search"]
        neural = name not in CLASSIC
        n = self.b["trials"] if neural else 1 if name == "reference" else self.b["comparator_trials"]
        rng = np.random.default_rng(self.seed)
        for i in range(n):
            if neural:
                h = dict(
                    branch=8 if i == 0 else int(rng.choice(grid["branch"])),
                    fusion=grid["fusion"][1 if i == 0 else rng.integers(3)],
                    lr=0.001 if i == 0 else float(np.exp(rng.uniform(*np.log(grid["lr"])))),
                    dropout=0.1 if i == 0 else float(rng.choice(grid["dropout"])),
                    decay=float(rng.choice(grid["decay"])),
                    batch=int(rng.choice(grid["batch"])),
                )
                h["lambda"] = grid["lambda"][i % len(grid["lambda"])]
            else:
                h = dict(
                    C=grid["comparator_C"][i],
                    ridge=grid["ridge"][i],
                    trees=12 if self.mode == "SMOKE" else grid["trees"][i],
                    depth=grid["depth"][i],
                )
            yield model_spec(name, h, task)

    def record(self, name, entry):
        p = self.log / (name + ".jsonl")
        with p.open("a", encoding="utf8") as f:
            f.write(json.dumps(entry, default=str) + "\n")

    def fit(self, task, name, tr, va, h, seed, ep, label):
        if self.mode == "AUDIT":
            raise ValueError("AUDIT disables training")
        if set(tr.cohort) != {self.sources(task)[0]}:
            raise ValueError("Source-only fit guard")
        if va is not None and (set(va.cohort) != set(tr.cohort) or set(va.id) & set(tr.id)):
            raise ValueError("Validation leakage")
        kind = name if name in CLASSIC else "net"
        ident = dict(
            **self.identity,
            model=name,
            task=task,
            fold=label,
            seed=seed,
            h=h,
            epochs=ep,
            train_keys=key(tr.id.tolist()),
            validation_keys=key(va.id.tolist()) if va is not None else None,
        )
        folder = self.private / "fits" / task / label / name / key(ident)[:16]
        final = folder / "complete.pkl"
        if final.exists():
            return restore(final)
        st = time.perf_counter()
        try:
            model, n = fit_model(
                kind, tr, va, h, task, seed, ep, self.b["patience"], self.device, folder / "epoch.pkl", ident
            )
            res = (model, n)
            dump(final, res)
            self.record(
                "fits",
                dict(
                    task=task,
                    model=name,
                    fold=label,
                    seed=seed,
                    epochs=n,
                    seconds=time.perf_counter() - st,
                    status="complete",
                    config=h,
                    parameters=sum((p.numel() for p in model.obj.parameters())) if kind == "net" else None,
                ),
            )
            return res
        except Exception as e:
            self.record(
                "fits",
                dict(
                    task=task,
                    model=name,
                    fold=label,
                    status="failed",
                    seconds=time.perf_counter() - st,
                    error=repr(e),
                    config=h,
                ),
            )
            raise

    def tune(self, task, name, d, label, fixed=None, epochs=None, grid=None):
        split = self.split(d, self.b["inner"], label + "_inner", self.seed + 11)
        if grid is not None:
            candidates = grid
        elif fixed is not None:
            candidates = [fixed]
        else:
            candidates = self.candidates(name, task)
        best = None
        for trial, h in enumerate(candidates):
            preds = np.full(len(d), np.nan)
            used = []
            scores = []
            allw = np.zeros(len(d))
            ally = np.zeros(len(d))
            for i, (a, b) in enumerate(split):
                tr, va = (d.iloc[a], d.iloc[b])
                try:
                    model, n = self.fit(
                        task,
                        name,
                        tr,
                        va if epochs is None else None,
                        h,
                        self.seed,
                        epochs or self.b["epochs"],
                        f"{label}_c{trial}_i{i}",
                    )
                    preds[b] = model.predict(va)
                    y, w, _ = weights(va, h.get("site_g", False))
                    allw[b], ally[b] = (w, y)
                    scores.append(brier(y, w, preds[b]) * len(b))
                    used.append(n)
                except (ValueError, RuntimeError, FloatingPointError, ArithmeticError) as e:
                    self.record(
                        "candidates",
                        dict(model=name, label=label, trial=trial, inner=i, status="failed", error=repr(e)),
                    )
                    break
            if np.isfinite(preds).all():
                score = sum(scores) / len(d)
                refit_epochs = epochs if epochs is not None else max(1, round(np.median(used)))
                config = dict(
                    h=h,
                    epochs=refit_epochs,
                    score=score,
                    threshold=point_threshold(ally, allw, preds),
                    trial=trial,
                )
                self.record("candidates", dict(model=name, label=label, status="scored", **config))
                if best is None or score < best["score"]:
                    best = config
        if best is None:
            raise RuntimeError(f"All candidates failed for {name} / {label}; see logs")
        return best

    def plan(self, task, names, d, label, variants=False):
        p = self.private / "plans" / (label + ".pkl")
        if p.exists():
            return restore(p)
        plans = {}
        for name in names:
            plans[name] = self.tune(task, name, d, label)
        if variants:
            anchor = plans["M3"]
            base = anchor["h"]
            epochs = anchor["epochs"]
            positive_lambdas = sorted({v for v in self.cfg["search"]["lambda"] if v > 0})
            if not positive_lambdas:
                raise ValueError("Matched penalty comparison requires a strictly positive lambda candidate")
            plans["fixed_zero"] = self.tune(
                task, "fixed_zero", d, label, fixed=dict(base, **{"lambda": 0.0}), epochs=epochs
            )
            grid = [dict(base, **{"lambda": v}) for v in positive_lambdas]
            plans["fixed_positive"] = self.tune(task, "fixed_positive", d, label, grid=grid, epochs=epochs)
            for name in PRIOR_VARIANTS + OTHER_VARIANTS:
                ref = plans["M1" if name == "nonlab_M1" else "M3"]
                if name in PRIOR_VARIANTS:
                    ref = plans["fixed_positive"]
                h = model_spec(name, ref["h"], task)
                plans[name] = self.tune(task, name, d, label, fixed=h, epochs=ref["epochs"])
        dump(p, plans)
        return plans

    def model_names(self, task):
        return [n for n in self.cfg["models"] if task == "A" or n != "Cox"]

    def internal(self, task="A", source=None, site=False):
        src = source or self.sources(task)[0]
        d = self.data(src)
        tag = self.tag(task)
        label = tag + "_" + src + ("_site" if site else "_nested")
        if site:
            splits = []
            for g in sorted(d.site.unique()):
                test = np.flatnonzero(d.site.to_numpy() == g)
                train = np.flatnonzero(d.site.to_numpy() != g)
                splits.append((train, test))
            sp = self.private / "splits" / (label + ".pkl")
            if not sp.exists():
                dump(sp, dict(ids=d.id.tolist(), folds=splits))
            else:
                splits = restore(sp)["folds"]
        else:
            splits = self.split(d, self.b["outer"], label, self.seed)
        variants = task == "A" and src == "RICH" and (not site) and self.cfg["options"]["ablations"]
        rows = []
        for i, (a, b) in enumerate(splits):
            out = self.private / "oof" / (label + f"_o{i}.pkl")
            if out.exists():
                rows += restore(out)
                continue
            tr, va = (d.iloc[a], d.iloc[b])
            names = self.model_names(task) if src == "RICH" and (not site) else ["M1", "M3"]
            lab = label + f"_o{i}"
            if site and (
                (status(va) == 1).sum() < self.cfg["min_cases"]
                or (status(va) == 0).sum() < self.cfg["min_controls"]
            ):
                rows.append(
                    dict(
                        fold=i,
                        site=str(va.site.iloc[0]),
                        model="all",
                        status="insufficient support",
                        n=len(b),
                        events=int((status(va) == 1).sum()),
                    )
                )
                continue
            print(f"{label}: fitting fold {i + 1}/{len(splits)}", flush=True)
            plans = self.plan(task, names, tr, lab, variants)
            result = []
            if src == "RICH" and self.cfg["options"]["site_ipcw"] and (not site):
                ref = plans["M3"]
                h = model_spec("site_ipcw", ref["h"], task)
                try:
                    plans["site_ipcw"] = dict(
                        self.tune(task, "site_ipcw", tr, lab, fixed=h, epochs=ref["epochs"]),
                        epochs=ref["epochs"],
                    )
                except RuntimeError as exc:
                    if not str(exc).startswith("All candidates failed"):
                        raise
                    self.record(
                        "diagnostics",
                        dict(task=task, fold=i, model="site_ipcw", status="unavailable", error=str(exc)),
                    )
                    result.append(dict(fold=i, model="site_ipcw", status="unavailable; see candidate log"))
            for n, plan in plans.items():
                model, _ = self.fit(task, n, tr, None, plan["h"], self.seed, plan["epochs"], lab + "_refit")
                start = time.perf_counter()
                pred = model.predict(va)
                seconds = time.perf_counter() - start
                dump(
                    self.private / "predictions" / lab / (n + ".pkl"),
                    dict(id=va.id.tolist(), p=pred, threshold=plan["threshold"]),
                )
                result.append(
                    dict(
                        fold=i,
                        model=n,
                        site=str(va.site.iloc[0]) if site else "mixed",
                        status="complete",
                        scoring_seconds=seconds,
                        **metrics(va, pred, plan["threshold"]),
                    )
                )
                if "site_ipcw" in plans and n in ("M3", "site_ipcw"):
                    result.append(
                        dict(
                            fold=i,
                            model=n + "_siteG_metric",
                            site="mixed",
                            status="complete",
                            scoring_seconds=seconds,
                            **metrics(va, pred, plan["threshold"], True),
                        )
                    )
                if n == "M1" and variants:
                    result.append(dict(result[-1], model="M2"))
            dump(out, result)
            rows += result
        csv(self.table / tag / (label + ".csv"), rows)
        return pd.DataFrame(rows)

    def final_plan(self, task):
        source = self.sources(task)[0]
        d = self.data(source)
        return self.plan(
            task,
            self.model_names(task),
            d,
            self.tag(task) + "_final",
            task == "A" and self.cfg["options"]["ablations"],
        )

    def freeze(self, task):
        tag = self.tag(task)
        directory = self.private / "frozen" / tag
        manifest = directory / "manifest.json"
        if manifest.exists():
            self.frozen(task)
            return manifest
        plans = self.final_plan(task)
        d = self.data(self.sources(task)[0])
        files = {}
        for name, c in plans.items():
            for seed in self.cfg["final_seed_pool"][: self.b["final_seeds"] if name not in CLASSIC else 1]:
                model, _ = self.fit(task, name, d, None, c["h"], seed, c["epochs"], tag + "_final_refit")
                p = directory / f"{name}_{seed}.pkl"
                dump(p, model)
                files[p.name] = digest(p)
        js(
            manifest,
            dict(
                identity=self.identity,
                source=self.sources(task)[0],
                plans=plans,
                files=files,
                ensemble="equal arithmetic mean of probabilities",
            ),
        )
        return manifest

    def frozen(self, task):
        folder = self.private / "frozen" / self.tag(task)
        m = json.loads((folder / "manifest.json").read_text())
        if m["identity"] != self.identity or any((digest(folder / n) != h for n, h in m["files"].items())):
            raise ValueError("Frozen identity mismatch")
        return (folder, m)

    def selection_table(self, task, frozen=False):
        plans = self.frozen(task)[1]["plans"] if frozen else self.final_plan(task)
        definitions = {
            "M1": "unpenalized primary baseline",
            "M3": "primary model, zero eligible",
            "fixed_zero": "matched M3 architecture, zero penalty",
            "fixed_positive": "matched M3 architecture, strictly positive penalty",
        }
        rows = []
        for name, plan in plans.items():
            neural = name not in CLASSIC
            base = plan["h"]
            rows.append(
                dict(
                    model=name,
                    lambda_value=base["lambda"] if neural else None,
                    source_cv_brier=plan["score"],
                    threshold=plan["threshold"],
                    candidate_index=plan["trial"],
                    epochs=plan["epochs"] if neural else None,
                    role=definitions.get(name, ""),
                    settings=json.dumps(base, sort_keys=True),
                )
            )
        frame = pd.DataFrame(rows)
        csv(self.table / self.tag(task) / "source_selection.csv", frame)
        return frame

    def score(self, task):
        tag = self.tag(task)
        folder, m = self.frozen(task)
        d = self.data(self.sources(task)[1])
        out = {}
        rows = []
        seeds = []
        for n, plan in m["plans"].items():
            files = [f for f in m["files"] if f.rsplit("_", 1)[0] == n]
            st = time.perf_counter()
            each = np.stack([restore(folder / f).predict(d) for f in files])
            p = each.mean(0)
            seconds = time.perf_counter() - st
            out[n] = p
            rows.append(dict(model=n, scoring_seconds=seconds, **metrics(d, p, plan["threshold"])))
            for f, pr in zip(files, each):
                seeds.append(dict(model=n, seedfile=f, **metrics(d, pr, plan["threshold"])))
        out["M2"] = out["M1"].copy()
        assert np.array_equal(out["M2"], out["M1"])
        rows.append(dict(next((x for x in rows if x["model"] == "M1")), model="M2"))
        dump(
            self.private / "predictions" / (tag + "_frozen.pkl"),
            dict(id=d.id.tolist(), p=out, manifest=digest(folder / "manifest.json")),
        )
        csv(self.table / tag / "external_metrics.csv", rows)
        csv(self.table / tag / "seed_metrics.csv", seeds)
        return pd.DataFrame(rows)

    def results(self, task):
        folder, m = self.frozen(task)
        d = self.data(self.sources(task)[1])
        obj = restore(self.private / "predictions" / (self.tag(task) + "_frozen.pkl"))
        if d.id.tolist() != obj["id"] or obj["manifest"] != digest(folder / "manifest.json"):
            raise ValueError("Frozen scoring mismatch")
        return (d, obj["p"], m)

## 08 · Package module `analysis.py`: post-freeze analyses

Paired external contrasts, calibration/ROC/PR figures, pooled out-of-fold summaries, subgroups with the 20-case support rule, integrated gradients with completeness checks, finite-step and analytical direction checks, ICE curves, optional random-forest SHAP, optional target intercept updating, the claim-to-evidence register and the aggregate export.

In [ ]:
%%writefile ckinn_obj1/analysis.py
from pathlib import Path
import json, zipfile
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.special import expit, logit
from sklearn.metrics import roc_curve, precision_recall_curve
from data import *
from stats import *
from models import DIRECTIONS
from runner import VARIANTS


def external_contrast(st, task="A", matched=False):
    d, p, m = st.results(task)
    if matched:
        candidate, baseline = ("fixed_positive", "fixed_zero")
        filename = "matched_positive_paired_brier.csv"
        left = m["plans"][candidate]
        right = m["plans"][baseline]
        left_base = {k: v for k, v in left["h"].items() if k != "lambda"}
        right_base = {k: v for k, v in right["h"].items() if k != "lambda"}
        if (
            left_base != right_base
            or left["epochs"] != right["epochs"]
            or (not left["h"]["lambda"] > 0)
            or (right["h"]["lambda"] != 0)
        ):
            raise ValueError("Frozen matched pair must differ only in penalty strength")
    else:
        candidate, baseline = ("M3", "M1")
        filename = "paired_brier.csv"
    y, w, _ = weights(d)
    delta = brier(y, w, p[candidate]) - brier(y, w, p[baseline])
    row = dict(
        comparison=f"{candidate} minus {baseline}",
        task=st.tag(task),
        lambda_candidate=m["plans"][candidate]["h"]["lambda"],
        lambda_baseline=m["plans"][baseline]["h"]["lambda"],
        n=len(d),
        known_cases=int(y.sum()),
        candidate_brier=brier(y, w, p[candidate]),
        baseline_brier=brier(y, w, p[baseline]),
        delta=delta,
        **bootstrap(d, p[candidate], p[baseline], st.b["bootstrap"], st.seed + 30),
    )
    csv(st.table / st.tag(task) / filename, [row])
    return row


def figures(st, task):
    d, p, m = st.results(task)
    y, w, _ = weights(d)
    ok = w > 0
    rows = []
    folder = st.fig / st.tag(task)
    folder.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")
    for n in ["M1", "M3"]:
        pred = p[n]
        cb = calbins(y, w, pred)
        cb["model"] = n
        rows += cb.to_dict("records")
        ax[0].plot(cb.predicted, cb.observed, "o-", label=n)
        if len(np.unique(y[ok])) == 2:
            f, t, _ = roc_curve(y[ok], pred[ok], sample_weight=w[ok])
            ax[1].plot(f, t, label=n)
            pr, re, _ = precision_recall_curve(y[ok], pred[ok], sample_weight=w[ok])
            ax[2].step(re, pr, where="post", label=n)
    lim = max(0.001, max(ax[0].get_xlim()[1], ax[0].get_ylim()[1]))
    ax[0].plot([0, lim], [0, lim], "--", color="gray")
    for a, (x, l) in zip(
        ax,
        [
            ("Mean predicted probability", "Weighted event fraction"),
            ("False-positive rate", "Sensitivity"),
            ("Recall", "Precision"),
        ],
    ):
        a.set(xlabel=x, ylabel=l)
        a.legend()
    fig.suptitle(f"{st.mode} · {st.tag(task)} · {st.sources(task)[1]}")
    fig.savefig(folder / "calibration_roc_pr.png", dpi=150)
    plt.close(fig)
    csv(st.table / st.tag(task) / "calibration_bins.csv", rows)


def oof(st, task, model):
    if model == "M2":
        model = "M1"
    source = st.sources(task)[0]
    d = st.data(source)
    lab = st.tag(task) + "_" + source + "_nested"
    rows = []
    for i in range(st.b["outer"]):
        o = restore(st.private / "predictions" / (lab + f"_o{i}") / (model + ".pkl"))
        rows.append(pd.DataFrame(dict(id=o["id"], p=o["p"], threshold=o["threshold"], fold=i)))
    df = d.merge(pd.concat(rows), on="id", validate="one_to_one")
    if len(df) != len(d):
        raise ValueError("Incomplete OOF coverage")
    return df


def internal_summary(st):
    tag = "A"
    folds = pd.read_csv(st.table / tag / "A_RICH_nested.csv")
    rows = []
    for n in folds.loc[folds.status == "complete", "model"].unique():
        if n.endswith("_siteG_metric"):
            continue
        try:
            d = oof(st, "A", n)
        except FileNotFoundError:
            rows.append(dict(model=n, status="incomplete outer-fold coverage"))
            continue
        rows.append(
            dict(
                model=n,
                status="computed; pooled OOF; evaluation G from RICH",
                **metrics(d, d.p.to_numpy(), d.threshold.to_numpy()),
            )
        )
    csv(st.table / tag / "pooled_oof.csv", rows)
    return pd.DataFrame(rows)


def subgroups(st, task):
    d, p, m = st.results(task)
    rows = []
    for model in ["M1", "M3"] + (["M4", "M5"] if "M5" in p else []):
        datasets = [("external", d, p[model], m["plans"][model]["threshold"])]
        if task == "A":
            od = oof(st, task, model)
            datasets.append(("development_OOF", od, od.p.to_numpy(), od.threshold.to_numpy()))
        for stage, dd, pr, threshold in datasets:
            cuts = {"all": np.ones(len(dd), bool)}
            for s in sorted(dd.sex.dropna().unique()):
                cuts[f"sex_code_{s:g}"] = dd.sex == s
            age = dd.age
            cuts.update(
                {"age_20_39": age < 40, "age_40_59": (age >= 40) & (age < 60), "age_60_plus": age >= 60}
            )
            if task == "A":
                cuts["FPG_below_5.6"] = dd.fpg < 5.6
                cuts["eight_all_observed"] = dd[CORE].notna().all(axis=1)
                cuts["eight_any_missing"] = dd[CORE].isna().any(axis=1)
                cuts["HDL_missing"] = dd.hdl.isna()
            if stage == "development_OOF":
                for site in sorted(dd.site.unique()):
                    cuts[f"site_{site}"] = dd.site == site
            for label, mask in cuts.items():
                pick = np.asarray(mask)
                part = dd.loc[pick].reset_index(drop=True)
                stat = status(part) if "time" in part else part.event.to_numpy()
                cases = int(sum(stat == 1))
                controls = int(sum(stat == 0))
                row = dict(
                    model=model,
                    stage=stage,
                    group=label,
                    n=len(part),
                    events=cases,
                    controls=controls,
                    unknown=int(sum(stat == -1)),
                )
                if cases < st.cfg["min_cases"] or controls < st.cfg["min_controls"]:
                    row["status"] = "insufficient known cases or controls"
                else:
                    th = threshold if np.ndim(threshold) == 0 else threshold[pick]
                    row.update(status="estimated", **metrics(part, pr[pick], th))
                    row.update(bootstrap(part, pr[pick], None, st.b["bootstrap"], st.seed + 31, th, True))
                rows.append(row)
    csv(st.table / st.tag(task) / "subgroups.csv", rows)
    frame = pd.DataFrame(rows)
    disparities = []
    for (model, stage), group in frame[frame.group.str.startswith("sex_code_")].groupby(["model", "stage"]):
        supported = group[group.status == "estimated"]
        result = dict(
            model=model,
            stage=stage,
            supported_categories=len(supported),
            status="insufficient paired support",
        )
        if len(supported) == 2:
            result["status"] = "descriptive absolute differences"
            for k in ["sensitivity", "specificity", "fpr", "fnr", "brier", "ece"]:
                if supported[k].notna().all():
                    result[k + "_gap"] = abs(supported[k].iloc[1] - supported[k].iloc[0])
        disparities.append(result)
    csv(st.table / st.tag(task) / "sex_gaps.csv", disparities)
    return frame


def grad_raw(models, d, feature):
    each = []
    for f in models:
        x = f.prep.transform(d)[0]
        j = f.prep.names.index(feature)
        t = torch.from_numpy(x).requires_grad_(True)
        f.obj.eval()
        grad = torch.autograd.grad(f.obj(t).sigmoid().sum(), t)[0][:, j].detach().numpy()
        each.append(grad / f.prep.scale[j])
    return np.mean(each, axis=0)


def probability(models, d):
    return np.mean([f.predict(d) for f in models], axis=0)


def ig(model, d, steps=64):
    pp = model.prep
    x = torch.tensor(pp.transform(d)[0])
    base = x.clone()
    base[:, : len(pp.names)] = torch.tensor((pp.med - pp.mu) / pp.scale, dtype=torch.float32)
    delta = x - base
    grads = torch.zeros_like(x)
    model.obj.eval()
    for i in range(steps):
        u = (base + delta * ((i + 0.5) / steps)).requires_grad_(True)
        grads += torch.autograd.grad(model.obj(u).sigmoid().sum(), u)[0].detach()
    a = (delta * grads / steps).detach().numpy()
    with torch.no_grad():
        basep = model.obj(base).sigmoid().numpy()
    return (a, basep)


def explain(st, task):
    tag = st.tag(task)
    folder, m = st.frozen(task)
    src = st.data(st.sources(task)[0])
    target = st.data(st.sources(task)[1])
    d = target.sample(min(512, len(target)), random_state=st.seed + 19).reset_index(drop=True)
    directions = []
    attributions = []
    ice_agg = []
    steps = {"fpg": 0.1, "bmi": 1.0, "age": 1.0, "sbp": 5.0, "tg": 0.1, "hdl": 0.1}
    figdir = st.fig / tag
    figdir.mkdir(parents=True, exist_ok=True)
    names = ["M1", "M3"] + [n for n in ("fixed_zero", "fixed_positive", "expanded_prior") if n in m["plans"]]
    for n in names:
        files = [f for f in m["files"] if f.rsplit("_", 1)[0] == n]
        nets = [restore(folder / f) for f in files]
        pp = nets[0].prep
        for role, models in [("frozen_average", nets)] + (
            [(f"seed_{f.rsplit('_', 1)[-1].split('.')[0]}", [net]) for f, net in zip(files, nets)]
            if len(nets) > 1
            else []
        ):
            for feature in (
                list(DIRECTIONS) if n == "expanded_prior" else ["bmi", "fpg"] if task == "A" else ["bmi"]
            ):
                j = pp.names.index(feature)
                step = steps[feature]
                eps = 0.005 * pp.scale[j]
                use = d[feature].between(pp.lo[j] + eps, pp.hi[j] - max(step, eps))
                dd = d.loc[use].copy()
                row = dict(
                    model=n,
                    deployment=role,
                    feature=feature,
                    raw_step=step,
                    n_sample=len(d),
                    n_eligible=len(dd),
                )
                if len(dd):
                    center = probability(models, dd)

                    def shifted(amount):
                        tmp = dd.copy()
                        tmp[feature] = dd[feature] + amount
                        return probability(models, tmp)

                    change = shifted(step) - center
                    sign = DIRECTIONS[feature]
                    deriv = grad_raw(models, dd, feature)
                    numeric = (shifted(eps) - shifted(-eps)) / (2 * eps)
                    err = np.abs(deriv - numeric) * pp.scale[j]
                    agree = np.allclose(deriv * pp.scale[j], numeric * pp.scale[j], atol=3e-05, rtol=0.015)
                    row.update(
                        violations=int(sum(sign * change < -1e-06)),
                        violation_rate=np.mean(sign * change < -1e-06),
                        mean_wrong_way_change=np.maximum(0, -sign * change).mean(),
                        meaningful_fraction=np.mean(abs(change) >= 0.001),
                        flat_fraction=np.mean(abs(change) <= 1e-06),
                        gradient_violation_rate=np.mean(sign * deriv * pp.scale[j] < -1e-06),
                        max_standardized_check_error=err.max(),
                        finite_gradient_check=bool(agree),
                    )
                    if not agree:
                        st.record("diagnostics", dict(task=task, **row))
                    if role == "frozen_average" and feature in ("fpg", "bmi"):
                        grid = np.linspace(pp.lo[j], pp.hi[j], 25)
                        records = dd.iloc[: min(24, len(dd))].copy()
                        curves = []
                        for g in grid:
                            test = records.copy()
                            test[feature] = g
                            curves.append(probability(models, test))
                        mat = np.array(curves)
                        fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
                        ax.plot(grid, mat, alpha=0.3, linewidth=1)
                        ax.set(
                            xlabel=f"{feature} ({UNITS[feature]})",
                            ylabel="Predicted probability",
                            title=f"{n} · individual sensitivity",
                        )
                        priv = st.private / "examples" / tag
                        priv.mkdir(parents=True, exist_ok=True)
                        fig.savefig(priv / f"{n}_{feature}_ICE.png", dpi=130)
                        plt.close(fig)
                        for g, v in zip(grid, mat):
                            ice_agg.append(dict(model=n, feature=feature, value=g, mean=v.mean(), n=len(v)))
                directions.append(row)
        arr, base = zip(*(ig(net, d) for net in nets))
        arr = np.mean(arr, axis=0)
        base = np.mean(base, axis=0)
        val = probability(nets, d)
        err = val - base - arr.sum(1)
        for i, feature in enumerate(pp.names):
            attributions.append(
                dict(
                    model=n,
                    feature=feature,
                    mean_abs_IG=abs(arr[:, i]).mean(),
                    mean_signed_IG=arr[:, i].mean(),
                    n=len(d),
                    steps=64,
                    max_abs_completeness_error=abs(err).max(),
                )
            )
        examples = pd.DataFrame(arr[:5, : len(pp.names)], columns=pp.names)
        examples.insert(0, "example", np.arange(1, len(examples) + 1))
        examples["baseline_probability"] = base[:5]
        examples["prediction"] = val[:5]
        examples["integration_residual"] = err[:5]
        csv(st.private / "examples" / tag / (n + "_IG.csv"), examples)
    if ice_agg:
        f = pd.DataFrame(ice_agg)
        fig, axs = plt.subplots(
            1, f.feature.nunique(), figsize=(5 * f.feature.nunique(), 4), squeeze=False, layout="constrained"
        )
        for ax, (feat, g) in zip(axs.ravel(), f.groupby("feature")):
            for n, h in g.groupby("model"):
                ax.plot(h.value, h["mean"], label=n)
            ax.set(xlabel=f"{feat} ({UNITS[feat]})", ylabel="Mean predicted probability")
            ax.legend()
        fig.suptitle(f"{st.mode} · {tag} · average raw-input perturbations")
        fig.savefig(figdir / "average_perturbation.png", dpi=150)
        plt.close(fig)
    for name, arr in [("directions", directions), ("attributions", attributions), ("perturbation", ice_agg)]:
        csv(st.table / tag / (name + ".csv"), arr)
    if st.cfg["options"]["tree_shap"]:
        import shap

        model = restore(folder / next((f for f in m["files"] if f.startswith("RF_"))))
        bg = model.prep.transform(src.sample(min(64, len(src)), random_state=st.seed))[0]
        vals = shap.TreeExplainer(
            model.obj, data=bg, feature_perturbation="interventional", model_output="probability"
        ).shap_values(model.prep.transform(d.iloc[:32])[0])
        a = np.asarray(vals)
        if isinstance(vals, list):
            a = np.asarray(vals[1])
        elif a.ndim == 3:
            a = a[:, :, 1]
        names = model.prep.names + [k + "_missing" for k in model.prep.names]
        csv(
            st.table / tag / "rf_shap.csv",
            pd.DataFrame(
                dict(feature=names, mean_abs_SHAP=abs(a).mean(0), background=len(bg), n=min(32, len(d)))
            ),
        )


def updating(st):
    d, p, m = st.results("A")
    th = m["plans"]["M3"]["threshold"]
    size = st.cfg["update_n"]
    if len(d) <= size:
        raise ValueError("Target too small for reserved update split")
    order = np.random.default_rng(st.seed + 66).permutation(len(d))
    a, b = (order[:size], order[size:])
    dump(
        st.private / "splits" / "target_update.pkl",
        dict(update=d.id.iloc[a].tolist(), evaluation=d.id.iloc[b].tolist()),
    )
    row = []
    dd = d.iloc[a]
    stat = status(dd)
    if sum(stat == 1) < st.cfg["min_cases"] or sum(stat == 0) < st.cfg["min_controls"]:
        row.append(
            dict(
                status="skipped for insufficient updating support",
                n=len(a),
                cases=int(sum(stat == 1)),
                controls=int(sum(stat == 0)),
            )
        )
    else:
        y, w, _ = weights(dd)
        offset = calibration(y, w, p["M3"][a])[0]
        after = expit(logit(np.clip(p["M3"][b], 1e-07, 1 - 1e-07)) + offset)
        for name, v in [("frozen", p["M3"][b]), ("intercept_updated", after)]:
            row.append(
                dict(
                    model=name,
                    status="estimated",
                    update_n=len(a),
                    update_events=int(sum(stat == 1)),
                    offset=offset,
                    **metrics(d.iloc[b], v, th),
                )
            )
    csv(st.table / "A" / "target_update.csv", row)
    return row


def evidence(st):
    rows = []
    for task in ["A", "B"]:
        tag = st.tag(task)
        table = st.table / tag
        for claim, file in [
            ("External predictive error and calibration", "external_metrics.csv"),
            ("Paired risk error difference", "paired_brier.csv"),
            ("Subgroup estimates conditional on support", "subgroups.csv"),
            ("Signed local clinical behavior", "directions.csv"),
            ("Model-attributed probability contributions", "attributions.csv"),
        ]:
            rows.append(
                dict(
                    task=tag,
                    claim=claim,
                    artifact=file,
                    status="computed" if (table / file).is_file() else "not run",
                    scope=(
                        "software check only"
                        if st.mode in ("SMOKE", "AUDIT")
                        else "extract-level screening" if task == "B" else "recorded detection by 3 years"
                    ),
                )
            )
        for claim in [
            "Clinical benefit",
            "Adjudicated subtype",
            "Clinician agreement",
            "Physiological dynamics",
            "Repeated prediabetes transitions",
        ]:
            rows.append(
                dict(
                    task=tag,
                    claim=claim,
                    status="unmeasured",
                    artifact="",
                    scope="requires independent evidence",
                )
            )
    csv(st.report / "claim_to_evidence.csv", rows)
    lines = [f"# Objective 1 · {st.mode}", "Reanalysis. Results below are read from this run's saved tables."]
    if st.mode in ("AUDIT", "SMOKE"):
        lines += ["Software and data checks only. These outputs support no study performance conclusion."]
    for tag in ["A", st.tag("B")]:
        if tag == "B_as_coded":
            lines.append(
                "The binary-zero screening subset has unresolved producer recoding for borderline and missing responses. Interpret it at the extract level."
            )
        p = st.table / tag / "paired_brier.csv"
        if p.exists():
            z = pd.read_csv(p).iloc[0]
            label = (
                "includes zero"
                if z.delta_lo <= 0 <= z.delta_hi
                else "lies below zero" if z.delta_hi < 0 else "lies above zero"
            )
            if not np.isfinite([z.delta_lo, z.delta_hi]).all():
                label = "is unavailable for this run"
            lines.append(
                f"## {tag}\n\nM3 minus M1 Brier difference {z.delta:.6g}. The paired 95% interval [{z.delta_lo:.6g}, {z.delta_hi:.6g}] {label}. Negative values favor M3. Uncertainty is conditional on these fitted models."
            )
            if z.lambda_candidate == 0:
                lines.append(
                    "M3 selected a zero clinical penalty. This comparison estimates performance under the primary tuning rule."
                )
        else:
            lines.append(f"## {tag}\n\nThe paired comparison has not run.")
        matched = st.table / tag / "matched_positive_paired_brier.csv"
        if matched.exists():
            result = pd.read_csv(matched).iloc[0]
            lines.append(
                f"The secondary matched comparison selected lambda {result.lambda_candidate:g} in RICH. Positive minus zero Brier difference is {result.delta:.6g}, with paired 95% bounds [{result.delta_lo:.6g}, {result.delta_hi:.6g}]. Negative values favor the positive-penalty fit."
            )
        ext = st.table / tag / "external_metrics.csv"
        if ext.exists():
            models = pd.read_csv(ext).set_index("model")
            for name in ["M1", "M3"]:
                r = models.loc[name]
                lines.append(
                    f"{name} had Brier {r.brier:.5g}, AUROC {r.auroc:.4g}, average precision {r.average_precision:.4g}, mean prediction {r.predicted:.4g}, and adjusted observed fraction {r.observed:.4g}. Calibration intercept {r.cal_intercept:.4g} and slope {r.cal_slope:.4g} use {int(r.n)} participants with {int(r.events)} known cases."
                )
        dirs = st.table / tag / "directions.csv"
        if dirs.exists():
            di = pd.read_csv(dirs)
            if "finite_gradient_check" in di:
                failed = di.finite_gradient_check.eq(False).sum()
                lines.append(
                    f"Analytical versus numerical direction checks recorded {failed} disagreements outside tolerance. See eligible denominators and flatness in directions.csv."
                )
        groups = st.table / tag / "subgroups.csv"
        if groups.exists():
            sg = pd.read_csv(groups)
            lines.append(
                f"{(sg.status != 'estimated').sum()} of {len(sg)} model-by-group estimates lacked the prespecified support."
            )
    lines.append(
        "Five- and seven-year hypotheses are inactive. Study claims require a scientific-profile run and review of support, calibration, missingness and fit failures."
    )
    atomic(st.report / "summary.md", lambda p: p.write_text("\n\n".join(lines), encoding="utf8"))
    js(
        st.report / "run_manifest.json",
        dict(
            mode=st.mode,
            run=st.run,
            identity=st.identity,
            options=st.cfg["options"],
            code=st.code,
            budget=st.b,
            artifacts=[str(p.relative_to(st.root)) for p in st.table.rglob("*.csv")],
            conditional_uncertainty="participant resampling of frozen predictions; separate seed file",
            frozen={
                q.name: digest(q / "manifest.json")
                for q in (st.private / "frozen").glob("*")
                if (q / "manifest.json").exists()
            },
        ),
    )


def export(st, package):
    evidence(st)
    dest = st.report / ("CKINN_" + st.mode + "_aggregate.zip")

    def write(path):
        with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
            for folder in [st.table, st.fig, st.report, st.auditpath]:
                for f in folder.rglob("*"):
                    if f.is_file() and f.suffix in (".csv", ".json", ".md", ".png"):
                        z.write(f, str(f.relative_to(st.root)))
            for f in Path(package).rglob("*"):
                if (
                    f.is_file()
                    and f.suffix in (".py", ".ipynb", ".txt", ".md", ".json", ".csv", ".png")
                    and (not f.is_relative_to(st.root / "data"))
                ):
                    z.write(f, "software/" + str(f.relative_to(package)))
            for f in st.log.glob("*.json*"):
                z.write(f, "logs/" + f.name)

    atomic(dest, write)
    return dest

## 09 · Package module `stages.py`: stage groups and display helpers

Defines the stage groups used by `RUN_STAGE`, the IPCW support table and small display helpers.

In [ ]:
%%writefile ckinn_obj1/stages.py
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image
from data import CORE, NONLAB, PLUS, UNITS, PERIODS, R, J, csv, js, restore, digest, key, allowed
from stats import support, ReverseKM
from models import Net, Prep, seed_all
from runner import VARIANTS, env_info
from analysis import (
    external_contrast,
    figures,
    subgroups,
    explain,
    updating,
    evidence,
    export,
    internal_summary,
)
from verify import run_checks, Checks
import unittest

GROUPS = {
    "AUDIT": list(range(5, 14)) + [33],
    "CHECKS": [14, 17, 20],
    "INTERNAL": [21, 22],
    "FINAL": [23, 24],
    "EXTERNAL": [25, 26, 27],
    "ABLATIONS": [28],
    "SENSITIVITY": [29],
    "EXPLAIN": [30],
    "SUBGROUPS": [31],
    "UPDATE": [32],
    "SCREEN": [33, 34, 35],
    "REPORT": [36, 37, 38, 39],
}
TRAIN_STAGES = set(range(21, 33)) | {34, 35}
SELECTION_COLS = ["model", "lambda_value", "epochs", "source_cv_brier", "candidate_index", "threshold"]
METRIC_COLS = [
    "model",
    "n",
    "events",
    "brier",
    "auroc",
    "average_precision",
    "predicted",
    "observed",
    "cal_slope",
    "sensitivity",
    "specificity",
]


def show(path):
    p = Path(path)
    if not p.exists():
        print(f"Unavailable in this run: {p.name}")
        return
    if p.suffix == ".csv":
        display(pd.read_csv(p).head(25))
    elif p.suffix == ".png":
        display(Image(filename=str(p)))
    else:
        print(p.read_text(encoding="utf8")[:5000])


def support_table(st):
    result = []
    for name in ["RICH", "NAGALA"]:
        d = st.data(name)
        for site, part in [("ALL", d)] + list(d.groupby("site")):
            row = dict(cohort=name, site=site)
            try:
                row.update(support(part), status="estimated")
            except ValueError as e:
                row.update(
                    n=len(part),
                    events=int(((part.event == 1) & (part.time <= 3)).sum()),
                    controls=int((part.time > 3).sum()),
                    unknown=int(((part.event == 0) & (part.time <= 3)).sum()),
                    G3=float(ReverseKM(part.time, part.event).at(3)),
                    at_risk_3=int((part.time >= 3).sum()),
                    status=str(e),
                )
            result.append(row)
    csv(st.table / "A" / "ipcw_support.csv", result)
    return pd.DataFrame(result)


def test_cases(names):
    suite = unittest.TestSuite((Checks(n) for n in names))
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise AssertionError("Stage check failed")


def run(n, st, package):
    opt = st.cfg["options"]
    stage = st.cfg["stage"]
    if stage != "ALL" and stage not in GROUPS:
        raise ValueError("Unknown RUN_STAGE: " + stage)
    if stage != "ALL" and n not in GROUPS[stage]:
        return
    if st.mode == "AUDIT" and n in TRAIN_STAGES:
        print("Skipped in AUDIT mode.")
        return
    if n == 5:
        print(pd.Series(st.identity["data"]).to_string())
        st.audit()
    elif n == 6:
        info = json.loads((st.auditpath / "inventory.json").read_text(encoding="utf8"))
        for sheet in info["workbooks"]:
            print(sheet)
        print(info["zip_members"], info["consistency_report"])
    elif n == 7:
        for f in ["structure", "raw_features", "formula_cache", "unit_checks"]:
            show(st.auditpath / (f + ".csv"))
    elif n == 8:
        display(pd.DataFrame([dict(input=k, RICH=R[k], NAGALA=J[k], unit=UNITS[k]) for k in PLUS]))
    elif n == 9:
        show(st.auditpath / "cohort_flow.csv")
    elif n == 10:
        display(support_table(st))
    elif n == 11:
        print("Primary inputs", allowed(CORE, "A"), "Screening", allowed(NONLAB, "B"))
        if st.mode != "AUDIT":
            st.split(st.data("RICH"), st.b["outer"], "A_RICH_nested", st.seed)
    elif n == 12:
        print(
            Prep.__doc__ or "Preprocessor loaded. Imputation and scale are fitted inside each training split."
        )
    elif n == 13:
        print("A case at exactly year 3 is included. A censoring at exactly year 3 has unknown status.")
    elif n == 14:
        test_cases(["test_boundaries_ties_hand_weights", "test_loss_denominator_and_penalty"])
    elif n == 15:
        print("The primary Brier denominator is the entire evaluation cohort.")
    elif n == 16:
        display(Image(filename=str(Path(package) / "architecture.png")))
    elif n == 17:
        test_cases(
            ["test_loss_denominator_and_penalty", "test_derivative_sign_masks_and_raw_finite_difference"]
        )
    elif n == 18:
        print(st.cfg["models"], "Comparator candidate budget", st.b["comparator_trials"])
    elif n == 19:
        print(st.b, "Resume directory", st.private)
    elif n == 20:
        js(st.report / "software_checks.json", run_checks())
    elif n == 21:
        display(st.internal().head())
    elif n == 22:
        show(st.table / "A" / "A_RICH_nested.csv")
        display(internal_summary(st))
    elif n == 23:
        display(st.selection_table("A")[SELECTION_COLS])
    elif n == 24:
        print(st.freeze("A"))
    elif n == 25:
        display(st.score("A")[METRIC_COLS])
    elif n == 26:
        display(pd.Series(external_contrast(st)))
        if opt["ablations"]:
            display(pd.Series(external_contrast(st, matched=True)))
    elif n == 27:
        figures(st, "A")
        show(st.fig / "A" / "calibration_roc_pr.png")
    elif n == 28:
        if opt["ablations"]:
            p = st.table / "A" / "external_metrics.csv"
            if not p.exists():
                raise RuntimeError("Complete FINAL and EXTERNAL before ABLATIONS")
            df = pd.read_csv(p)
            if not set(VARIANTS).issubset(df.model):
                raise ValueError("Component fits missing from frozen manifest")
            csv(st.table / "A" / "ablations.csv", df[df.model.isin(VARIANTS + ["M1", "M2", "M3"])])
            display(pd.read_csv(st.table / "A" / "ablations.csv")[METRIC_COLS])
        else:
            print("Component analyses disabled in this run.")
    elif n == 29:
        for option, src, site in [
            ("site_holdout", "RICH", True),
            ("broader_rich", "RICH_broad", False),
            ("alt_zero", "RICH_alt0", False),
        ]:
            if opt[option]:
                display(st.internal(source=src, site=site).head())
            else:
                print("Disabled", option)
        print("Site-IPCW sensitivity is fitted in stage 21 when enabled:", opt["site_ipcw"])
    elif n == 30:
        if opt["explain"]:
            explain(st, "A")
            show(st.table / "A" / "directions.csv")
        else:
            print("Explanations disabled.")
    elif n == 31:
        if opt["subgroups"]:
            display(subgroups(st, "A").head())
        else:
            print("Subgroups disabled.")
    elif n == 32:
        if opt["target_update"]:
            display(pd.DataFrame(updating(st)))
        else:
            print("Target updating disabled.")
    elif n == 33:
        show(st.auditpath / "cohort_flow.csv")
        print(
            "NHANES analysis label",
            st.tag("B"),
            "Producer recode evidence:",
            opt["producer_recode_evidence"] or "unavailable",
        )
    elif n == 34:
        if opt["screening"] == "off":
            print("Screening disabled.")
        else:
            print(st.freeze("B"))
            display(st.selection_table("B", frozen=True)[SELECTION_COLS])
    elif n == 35:
        if opt["screening"] == "off":
            print("Screening disabled.")
        else:
            display(st.score("B")[METRIC_COLS])
            display(pd.Series(external_contrast(st, "B")))
            figures(st, "B")
            if opt["subgroups"]:
                subgroups(st, "B")
            if opt["explain"]:
                explain(st, "B")
    elif n == 36:
        for p in sorted(st.table.rglob("*.csv")):
            print(p.relative_to(st.table))
        if (st.log / "fits.jsonl").exists():
            fits = pd.read_json(st.log / "fits.jsonl", lines=True)
            csv(
                st.table / "fit_budget.csv",
                fits.groupby(["task", "model", "status"], dropna=False)
                .agg(fits=("model", "size"), seconds=("seconds", "sum"))
                .reset_index(),
            )
    elif n == 37:
        evidence(st)
        show(st.report / "summary.md")
    elif n == 38:
        evidence(st)
        show(st.report / "run_manifest.json")
    elif n == 39:
        print(export(st, package))

## 10 · Package module `verify.py`: software checks

Unit checks on small artificial arrays only (no cohort rows):

* hand-computed weights with ties;
* the full-N Brier denominator;
* penalty gradients and mask exclusion;
* raw-unit finite differences;
* integrated-gradient completeness;
* training-only preprocessing and forbidden predictors;
* unit conversion and eligibility flow;
* ZIP safety;
* the matched zero-penalty identity;
* exact epoch resumption;
* the estimator weight response;
* site weights.

The integration routine runs the whole pipeline on artificial records.

In [ ]:
%%writefile ckinn_obj1/verify.py
import copy, inspect, json, tempfile, zipfile, unittest, sys
from pathlib import Path
from unittest.mock import patch
import numpy as np
import pandas as pd
import torch
from data import *
from models import *
from stats import *
from runner import Study, model_spec
from analysis import grad_raw, ig, external_contrast, figures, subgroups, explain, updating, evidence


def fake(n=120, cohort="RICH", task="A", seed=42):
    r = np.random.default_rng(seed)
    d = pd.DataFrame(
        dict(
            age=r.integers(20, 76, n),
            sex=r.integers(0, 2, n),
            bmi=r.normal(25, 3, n),
            sbp=r.normal(120, 10, n),
            alt=r.uniform(10, 60, n),
            fpg=r.uniform(4, 6, n),
            tg=r.uniform(0.5, 3, n),
            hdl=r.uniform(0.6, 1.9, n),
            dbp=r.normal(75, 5, n),
            smoking=r.integers(0, 2, n),
            tc=r.normal(5, 0.5, n),
            ast=r.uniform(12, 45, n),
            site=np.arange(n) % 3,
            cohort=cohort,
            id=[f"{cohort}-{i}" for i in range(n)],
        )
    )
    d["event"] = np.tile([1, 0, 0, 1, 0, 0], n // 6 + 1)[:n]
    if task == "A":
        d["time"] = np.tile([2, 2, 4, 5, 5, 1], n // 6 + 1)[:n]
    d.loc[d.index[::9], "hdl"] = np.nan
    return d


H = dict(
    features=CORE,
    masks=True,
    branch=8,
    fusion=[32, 16],
    lr=0.001,
    decay=0.0001,
    batch=24,
    dropout=0.1,
    directions=["fpg", "bmi"],
)
H["lambda"] = 10.0


class Checks(unittest.TestCase):

    def test_boundaries_ties_hand_weights(self):
        d = pd.DataFrame(dict(time=[1, 1, 2, 3, 3, 4, 5], event=[1, 0, 0, 1, 0, 1, 0]))
        y, w, g = weights(d)
        np.testing.assert_allclose(w, [1, 0, 0, 1.5, 0, 2.25, 2.25])
        np.testing.assert_equal(status(d), [1, -1, -1, 1, -1, 0, 0])
        self.assertAlmostEqual(g, 4 / 9)
        self.assertAlmostEqual(ReverseKM(d.time, d.event).at(3, True), 2 / 3)

    def test_loss_denominator_and_penalty(self):
        y = np.array([1.0, 0.0, 0.0])
        w = np.array([2.0, 1.5, 0.0])
        self.assertAlmostEqual(brier(y, w, np.array([0.5, 0.5, 0.9])), 0.875 / 3)
        pp = Prep(CORE, "A").fit(fake())
        x = torch.zeros((3, 16), requires_grad=True)
        inside = torch.ones((3, 8), dtype=torch.bool)
        a = torch.tensor(-0.6, requires_grad=True)
        h = dict(H, directions=["bmi"])

        def objective(lam):
            logits = a * x[:, 2]
            return loss_terms(
                logits,
                x,
                inside,
                torch.tensor(y),
                torch.tensor(w),
                torch.ones(3),
                pp,
                dict(h, **{"lambda": lam}),
            )

        np.testing.assert_allclose(objective(0).item(), 3.5 * np.log(2) / 3)
        g0 = torch.autograd.grad(objective(0), a)[0]
        g1 = torch.autograd.grad(objective(3), a)[0]
        self.assertNotEqual(g0.item(), g1.item())
        inside[:, 2] = False
        self.assertAlmostEqual(objective(3).item(), objective(0).item())

    def test_derivative_sign_masks_and_raw_finite_difference(self):
        d = fake()
        pp = Prep(CORE, "A").fit(d)
        x = torch.tensor(pp.transform(d)[0], requires_grad=True)
        p = (x[:, 7] * 0.5).sigmoid()
        ok = torch.from_numpy(pp.transform(d)[1])
        observed = p.detach().numpy()
        expected = np.mean((0.5 * observed[ok[:, 7]] * (1 - observed[ok[:, 7]])) ** 2)
        self.assertAlmostEqual(clinical(p, x, ok, pp.names, ["hdl"]).item(), expected, places=6)
        seed_all(27)
        net = Fitted("net", pp, Net(pp, H).eval())
        dd = d[d.hdl.notna()].copy()
        up = dd.copy()
        down = dd.copy()
        up["hdl"] += 0.002
        down["hdl"] -= 0.002
        np.testing.assert_allclose(
            (net.predict(up) - net.predict(down)) / 0.004, grad_raw([net], dd, "hdl"), atol=4e-05
        )
        a, base = ig(net, dd.iloc[:4])
        np.testing.assert_allclose(base + a.sum(1), net.predict(dd.iloc[:4]), atol=2e-05)

    def test_training_preprocessing_and_forbidden(self):
        d = fake()
        tr = d.iloc[:90]
        va = d.iloc[90:].copy()
        va["bmi"] = 8000
        pp = Prep(CORE, "A").fit(tr)
        np.testing.assert_allclose(pp.med[2], tr.bmi.median())
        pp.transform(va)
        np.testing.assert_allclose(pp.med[2], tr.bmi.median())
        x, ok = pp.transform(d)
        self.assertEqual(x[0, 15], 1)
        self.assertFalse(ok[0, 7])
        for task, bad in [
            ("A", ["event", "time", "site", "id", "FPG of final visit(mmol/L)"]),
            ("B", ["a1c", "HbA1c", "fpg", "FastingGlucose", "Diabetes_self", "Diabetes_lab"]),
        ]:
            for name in bad:
                with self.assertRaises(ValueError):
                    Prep(["age", name], task)
        for a, b in split_indices(d, 3, 11):
            self.assertFalse(set(d.id.iloc[a]) & set(d.id.iloc[b]))

    def test_schema_units_and_flow(self):
        src = pd.DataFrame({k: [10.0, 11.0, 12.0, 13.0] for k in J.values()})
        src[J["age"]] = [19, 45, 50, 55]
        src[J["sex"]] = [1, 0, 1, 0]
        src[J["time"]] = [365.25, 730.5, 1095.75, 1461.0]
        src[J["event"]] = [0, 1, 3, 0]
        src[J["a1c"]] = [5, 5.5, 5.5, 6.5]
        for k, (f, m) in CONVERT.items():
            src[f] = 90.0
            src[J[k]] = 90 * m
        src.loc[1, J["hdl"]] = 0
        d, checks = harmonize(src, "NAGALA")
        self.assertEqual(d.time.tolist(), [1, 2, 3, 4])
        self.assertAlmostEqual(d.hdl.iloc[1], 90 * 0.02586)
        self.assertEqual(d.sex.tolist(), [1, 0, 1, 0])
        good, lg = eligible(d)
        self.assertEqual(len(good), 1)
        self.assertEqual(good.event.iloc[0], 1)
        self.assertEqual([x["n"] for x in lg], [4, 3, 3, 2, 2, 1, 1])
        with self.assertRaises(ValueError):
            mapped(src.drop(columns=J["event"]), J)
        nh = pd.DataFrame(
            dict(
                Age=[20, 19, 40, 52],
                Sex=[0, 1, 1, 0],
                BMI=[24] * 4,
                SystolicBP=[115] * 4,
                HbA1c=[6.5, 7, 6.4, np.nan],
                Diabetes_self=[0, 0, 1, 0],
            )
        )
        s, fl = screen_candidates(nh, "period", 0)
        self.assertEqual((len(s), s.event.iloc[0]), (1, 1))

    def test_integrity_and_zip(self):
        with tempfile.TemporaryDirectory() as td:
            f = Path(td) / "x.pkl"
            dump(f, {"a": 17})
            self.assertEqual(restore(f), {"a": 17})
            f.write_bytes(f.read_bytes()[:-1])
            with self.assertRaises(ValueError):
                restore(f)
            z = Path(td) / "z.zip"
            for name in ["../esc.csv", "a/../../esc", "C:/esc", "/esc"]:
                with zipfile.ZipFile(z, "w") as a:
                    a.writestr(name, "x")
                with self.assertRaises(ValueError):
                    safe_unpack(z, Path(td) / "inside")

    def test_matched_zero_and_parameter_count(self):
        d = fake(60)
        pp = Prep(CORE, "A").fit(d)
        spec1 = model_spec("M1", dict(H), "A")
        spec0 = model_spec("fixed_zero", dict(H), "A")
        self.assertEqual(spec0, spec1)
        a, _ = train_net(d, None, pp, spec1, 10, 1, 2, "cpu", None, {})
        b, _ = train_net(d, None, pp, spec0, 10, 1, 2, "cpu", None, {})
        self.assertEqual(sum((p.numel() for p in a.parameters())), 1497)
        np.testing.assert_array_equal(torch_risk(a, pp.transform(d)[0]), torch_risk(b, pp.transform(d)[0]))

    def test_epoch_resume_identity(self):
        d = fake(60)
        pp = Prep(CORE, "A").fit(d)
        kwargs = dict(
            d=d,
            valid=None,
            prep=pp,
            h=H,
            seed=73,
            epochs=3,
            patience=2,
            device="cpu",
            identity={"fold": "artificial"},
        )
        full, _ = train_net(**kwargs, checkpoint=None)
        with tempfile.TemporaryDirectory() as td:
            f = Path(td) / "epoch.pkl"
            with self.assertRaises(InterruptedError):
                train_net(**kwargs, checkpoint=f, interrupt_epoch=1)
            with self.assertRaises(ValueError):
                train_net(**dict(kwargs, identity={"fold": "wrong"}), checkpoint=f)
            resumed, _ = train_net(**kwargs, checkpoint=f)
            for a, b in zip(full.parameters(), resumed.parameters()):
                self.assertTrue(torch.equal(a, b))

    def test_estimator_weight_response(self):
        d = fake(120)
        X = Prep(CORE, "A").fit(d).transform(d)[0]
        y = np.arange(len(d)) % 2
        h = dict(C=1.0, trees=10, depth=3)
        for n in ["LR", "RF", "XGB", "LGB", "SVM"]:

            def pred(w):
                m = fit_weighted(estimator(n, h, 10), X, y, w)
                return m.decision_function(X) if n == "SVM" else m.predict_proba(X)[:, 1]

            self.assertGreater(np.max(abs(pred(np.ones(len(y))) - pred(1 + 8 * y))), 1e-05, n)

    def test_site_weights_and_zero_support(self):
        d = fake()
        self.assertAlmostEqual(side_weights(d).mean(), 1)
        with self.assertRaises(ValueError):
            weights(d.assign(time=1.0))
        d["site"] = np.where(d.index < 5, "rare", "main")
        self.assertTrue(np.isfinite(weights(d, True)[1]).all())


def integration():
    import runner
    from analysis import export, internal_summary

    root0 = Path(__file__).resolve().parent
    cfg = json.loads((root0 / "config.json").read_text())
    cfg["profile"] = "SMOKE"
    cfg["device"] = "cpu"
    cfg["profiles"]["SMOKE"].update(epochs=1, bootstrap=2)
    cfg["profiles"]["SMOKE"]["trials"] = 1
    cfg["min_cases"] = 3
    cfg["min_controls"] = 3
    cfg["update_n"] = 60
    cfg["options"]["ablations"] = True
    with tempfile.TemporaryDirectory() as td:
        root = Path(td)
        ident = root / "synthetic_input.txt"
        ident.write_text("artificial only")
        st = Study(root, cfg, paths={"artificial": ident})
        for name, task, s in [
            ("RICH", "A", 7),
            ("NAGALA", "A", 14),
            (PERIODS[0], "B", 21),
            (PERIODS[1], "B", 28),
        ]:
            st.mem[name] = fake(180, name, task, s)
        for task in ["A", "B"]:
            if task == "A":
                st.internal(task)
                internal_summary(st)
            plans = st.final_plan(task)
            assert plans["M3"]["h"]["lambda"] == 0
            st.freeze(task)
            with patch.object(
                runner, "fit_model", side_effect=AssertionError("external scoring attempted training")
            ), patch.object(Prep, "fit", side_effect=AssertionError("external preprocessing refit")):
                st.score(task)
            d, p, _ = st.results(task)
            np.testing.assert_array_equal(p["M1"], p["M2"])
            selfsame = st.frozen(task)[1]
            assert len(d) == 180 and len(selfsame["files"]) >= 7
            external_contrast(st, task)
            if task == "A":
                zero, positive = [plans[n] for n in ["fixed_zero", "fixed_positive"]]
                assert zero["h"]["lambda"] == 0 and positive["h"]["lambda"] > 0
                assert zero["epochs"] == positive["epochs"]
                assert {k: v for k, v in positive["h"].items() if k != "lambda"} == {
                    k: v for k, v in zero["h"].items() if k != "lambda"
                }
                assert plans["expanded_prior"]["h"]["lambda"] == positive["h"]["lambda"]
                contrast = external_contrast(st, matched=True)
                assert contrast["delta_valid_boot"] == cfg["profiles"]["SMOKE"]["bootstrap"]
                assert (
                    abs(contrast["delta"] - (contrast["candidate_brier"] - contrast["baseline_brier"]))
                    < 1e-14
                )
                assert not np.array_equal(p["fixed_zero"], p["fixed_positive"])
            st.selection_table(task, frozen=True)
            figures(st, task)
            subgroups(st, task)
            explain(st, task)
        updating(st)
        for name in ["RICH_broad", "RICH_alt0"]:
            st.mem[name] = st.mem["RICH"].copy()
        st.internal("A", source="RICH_broad")
        st.internal("A", source="RICH_alt0")
        st.internal("A", site=True)
        out = export(st, root0)
        with zipfile.ZipFile(out) as z:
            assert not any(("checkpoints/" in n or n.endswith(".pkl") for n in z.namelist()))
    return "passed synthetic nested, frozen and matched positive penalty comparisons, subgroup, explanation, target update and export paths"


def run_checks():
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(Checks)
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise AssertionError("Software checks failed")
    return {"passed": result.testsRun, "cohort_fits": 0}


if __name__ == "__main__":
    print(run_checks())
    if "--integration" in sys.argv:
        print(integration())

## 11 · Package module `extension.py`: extension analyses (EXT, exploratory)

These analyses were specified after the first full development run; their rules are in `EXT_PRESPECIFICATION.md`.

* E2: penalty activity (wrong-way gradient shares and penalty versus data loss).
* E3 to E5: paired Brier differences against every baseline, the index of prediction accuracy, and IPCW net benefit.
* E6: NAGALA restricted to at least two years of follow-up, as in the RICH source design.
* E7: NAGALA with HDL treated as missing.
* E8: six-direction finite-step checks.
* E9, E9b: a penalty-strength path for six clinical directions, with paired contrasts.
* E10: a training-size curve on RICH subsamples.
* E11: RICH without sites lacking three-year support.
* E12: a paired calibration contrast.
* A generated table mapping the proposal's research questions and hypotheses to the computed estimates.

In [ ]:
%%writefile ckinn_obj1/extension.py
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from data import CORE, csv, dump, restore, status
from stats import weights, metrics, brier, calibration
from models import DIRECTIONS
from analysis import grad_raw, probability
from runner import CLASSIC

EXP_DIRS = ["fpg", "bmi", "age", "sbp", "tg", "hdl"]
LAMBDA_PATH = [0.0, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0]
SIZES = [1000, 3000, 10000, 30000]
DRAWS = [1, 2, 3, 4, 5]
SIZE_LAMBDAS = [10.0, 1000.0, 100000.0]
THRESHOLDS_NB = [0.005, 0.01, 0.02, 0.03, 0.05]
STEPS = {"fpg": 0.1, "bmi": 1.0, "age": 1.0, "sbp": 5.0, "tg": 0.1, "hdl": 0.1}
NEURAL = ["M1", "M3", "fixed_zero", "fixed_positive", "expanded_prior"]


def out(st):
    p = st.table / "A" / "EXT"
    p.mkdir(parents=True, exist_ok=True)
    return p


def n_boot(st):
    return st.b["bootstrap"] if st.mode == "SMOKE" else 1000


def plans_from_selection(selection_csv):
    sel = pd.read_csv(selection_csv)
    plans = {}
    for r in sel.itertuples():
        h = json.loads(r.settings)
        neural = r.model not in CLASSIC
        plans[r.model] = dict(
            h=h,
            epochs=int(r.epochs) if neural else 1,
            score=float(r.source_cv_brier),
            threshold=float(r.threshold),
            trial=int(r.candidate_index),
        )
    return plans


def compare_external(st, reference_csv):
    mine = pd.read_csv(st.table / "A" / "external_metrics.csv").set_index("model")
    ref = pd.read_csv(reference_csv).set_index("model")
    rows = []
    for m in ref.index:
        if m not in mine.index:
            rows.append(dict(model=m, status="missing in this run"))
            continue
        db, da = (
            abs(mine.loc[m, "brier"] - ref.loc[m, "brier"]),
            abs(mine.loc[m, "auroc"] - ref.loc[m, "auroc"]),
        )
        rows.append(
            dict(
                model=m,
                brier_this=mine.loc[m, "brier"],
                brier_reference=ref.loc[m, "brier"],
                abs_diff_brier=db,
                auroc_this=mine.loc[m, "auroc"],
                auroc_reference=ref.loc[m, "auroc"],
                abs_diff_auroc=da,
                status="match" if db < 1e-06 and da < 1e-06 else "MISMATCH",
            )
        )
    csv(out(st) / "E1_reconstruction_check.csv", rows)
    return pd.DataFrame(rows)


def frozen_models(st):
    folder, m = st.frozen("A")
    models = {n: [restore(folder / f) for f in m["files"] if f.rsplit("_", 1)[0] == n] for n in m["plans"]}
    return (models, m)


def prob_gradients(models, d):
    f0 = models[0]
    X, inside = f0.prep.transform(d)
    g = np.zeros((len(d), len(f0.prep.names)))
    for f in models:
        f.obj.eval()
        for idx in np.array_split(np.arange(len(d)), max(1, len(d) // 20000)):
            x = torch.from_numpy(X[idx]).requires_grad_(True)
            g[idx] += (
                torch.autograd.grad(f.obj(x).sigmoid().sum(), x)[0][:, : len(f0.prep.names)].detach().numpy()
            )
    return (g / len(models), inside, f0.prep.names)


def weighted_bce(d, p):
    y, w, _ = weights(d)
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(np.mean(w * -(y * np.log(p) + (1 - y) * np.log(1 - p))))


def penalty_diagnostics(st):
    models, m = frozen_models(st)
    rows = []
    for cohort in ["RICH", "NAGALA"]:
        d = st.data(cohort)
        for n in [x for x in NEURAL if x in models]:
            g, inside, names = prob_gradients(models[n], d)
            p = probability(models[n], d)
            h = m["plans"][n]["h"]
            penalised = h.get("directions", []) if h["lambda"] else []
            pen = []
            for f in EXP_DIRS:
                j = names.index(f)
                ok = inside[:, j]
                s = DIRECTIONS[f]
                term = np.maximum(0, -s * g[ok, j]) ** 2
                if f in penalised:
                    pen.append(term)
                rows.append(
                    dict(
                        cohort=cohort,
                        model=n,
                        lambda_value=h["lambda"],
                        feature=f,
                        penalised=f in penalised,
                        eligible_pairs=int(ok.sum()),
                        wrong_way_share=float(np.mean(s * g[ok, j] < 0)),
                        median_abs_dp_dz=float(np.median(abs(g[ok, j]))),
                        mean_squared_wrong_way=float(term.mean()),
                    )
                )
            data_loss = weighted_bce(d, p)
            pen_value = float(np.concatenate(pen).mean()) if pen else 0.0
            rows.append(
                dict(
                    cohort=cohort,
                    model=n,
                    lambda_value=h["lambda"],
                    feature="ALL_PENALISED",
                    penalised=True,
                    weighted_data_loss=data_loss,
                    clinical_penalty=pen_value,
                    lambda_times_penalty=h["lambda"] * pen_value,
                    ratio_to_data_loss=h["lambda"] * pen_value / data_loss,
                )
            )
    csv(out(st) / "E2_penalty_diagnostics.csv", rows)
    return pd.DataFrame(rows)


def _nb(y, w, p, t, n):
    s = p >= t
    return (np.sum(w * y * s) - t / (1 - t) * np.sum(w * (1 - y) * s)) / n


def external_comparisons(st):
    d, p, m = st.results("A")
    names = [n for n in m["plans"] if n in p]
    ref = "reference"
    y, w, _ = weights(d)
    N = len(d)

    def stats_of(y, w, idx):
        b = {n: float(np.mean(w * (y - p[n][idx]) ** 2)) for n in names}
        nb = {(n, t): _nb(y, w, p[n][idx], t, len(idx)) for n in ["M1", "M3", "LR"] for t in THRESHOLDS_NB}
        for t in THRESHOLDS_NB:
            nb["treat_all", t] = (np.sum(w * y) - t / (1 - t) * np.sum(w * (1 - y))) / len(idx)
        return (b, nb)

    b0, nb0 = stats_of(y, w, np.arange(N))
    rng = np.random.default_rng(st.seed + 40)
    draws_b, draws_nb, failed = ([], [], 0)
    B = n_boot(st)
    for _ in range(B):
        i = rng.integers(N, size=N)
        dd = d.iloc[i].reset_index(drop=True)
        try:
            yy, ww, _ = weights(dd)
            if np.unique(yy[ww > 0]).size < 2:
                raise ValueError("single class")
            bb, nn = stats_of(yy, ww, i)
            draws_b.append(bb)
            draws_nb.append(nn)
        except (ValueError, FloatingPointError):
            failed += 1
    DB = pd.DataFrame(draws_b)
    ok = len(DB) >= max(2, 0.8 * B)

    def ci(v):
        return tuple(np.quantile(v, [0.025, 0.975])) if ok else (np.nan, np.nan)

    rows = []
    for a in ["M1", "M3"]:
        for x in [n for n in ["reference", "LR", "RF", "XGB", "LGB", "SVM", "Cox"] if n in names]:
            lo, hi = ci(DB[a] - DB[x])
            rows.append(
                dict(
                    comparison=f"{a} minus {x}",
                    delta_brier=b0[a] - b0[x],
                    lo=lo,
                    hi=hi,
                    valid_boot=len(DB),
                    requested_boot=B,
                    failed_boot=failed,
                )
            )
    csv(out(st) / "E3_paired_brier_vs_baselines.csv", rows)
    rows = []
    for n in names:
        ipa = 1 - DB[n] / DB[ref]
        lo, hi = ci(ipa)
        rows.append(
            dict(
                model=n,
                brier=b0[n],
                brier_reference=b0[ref],
                ipa=1 - b0[n] / b0[ref],
                lo=lo,
                hi=hi,
                valid_boot=len(DB),
            )
        )
    csv(out(st) / "E4_index_of_prediction_accuracy.csv", rows)
    NB = pd.DataFrame([{f"{k[0]}|{k[1]}": v for k, v in x.items()} for x in draws_nb])
    rows = []
    for (n, t), v in nb0.items():
        lo, hi = ci(NB[f"{n}|{t}"])
        rows.append(dict(model=n, threshold=t, net_benefit=v, lo=lo, hi=hi))
    for t in THRESHOLDS_NB:
        rows.append(dict(model="treat_none", threshold=t, net_benefit=0.0, lo=0.0, hi=0.0))
    csv(out(st) / "E5_net_benefit.csv", rows)
    return True


def _score_all(st, d, label):
    models, m = frozen_models(st)
    rows = []
    for n, fs in models.items():
        pr = probability(fs, d)
        rows.append(dict(analysis=label, model=n, **metrics(d, pr, m["plans"][n]["threshold"])))
    return (rows, models)


def aligned_and_hdl(st):
    nag = st.data("NAGALA")
    aligned = nag[nag.time >= 2].reset_index(drop=True)
    rows, models = _score_all(st, aligned, "NAGALA follow-up >= 2 years (RICH-like)")
    csv(out(st) / "E6_estimand_aligned_nagala.csv", rows)
    nohdl = nag.assign(hdl=np.nan)
    rows, _ = _score_all(st, nohdl, "NAGALA with HDL set to missing")
    base = {n: probability(fs, nag) for n, fs in models.items()}
    for r in rows:
        q = probability(models[r["model"]], nohdl)
        r["mean_risk_change_vs_observed_hdl"] = float(np.mean(q - base[r["model"]]))
        r["mean_abs_risk_change"] = float(np.mean(abs(q - base[r["model"]])))
    csv(out(st) / "E7_nagala_hdl_missing.csv", rows)
    return True


def directions_all(st):
    models, m = frozen_models(st)
    target = st.data("NAGALA")
    d = target.sample(min(512, len(target)), random_state=st.seed + 19).reset_index(drop=True)
    rows = []
    for n in [x for x in ["M1", "M3", "fixed_positive", "expanded_prior"] if x in models]:
        fs = models[n]
        pp = fs[0].prep
        for f in EXP_DIRS:
            j = pp.names.index(f)
            step, eps = (STEPS[f], 0.005 * pp.scale[j])
            dd = d.loc[d[f].between(pp.lo[j] + eps, pp.hi[j] - max(step, eps))].copy()
            if not len(dd):
                continue
            center = probability(fs, dd)
            change = probability(fs, dd.assign(**{f: dd[f] + step})) - center
            s = DIRECTIONS[f]
            rows.append(
                dict(
                    model=n,
                    lambda_value=m["plans"][n]["h"]["lambda"],
                    feature=f,
                    penalised=f in m["plans"][n]["h"].get("directions", [])
                    and m["plans"][n]["h"]["lambda"] > 0,
                    raw_step=step,
                    n_eligible=len(dd),
                    violation_rate=float(np.mean(s * change < -1e-06)),
                    mean_wrong_way_change=float(np.maximum(0, -s * change).mean()),
                    mean_change=float(change.mean()),
                    meaningful_fraction=float(np.mean(abs(change) >= 0.001)),
                    flat_fraction=float(np.mean(abs(change) <= 1e-06)),
                )
            )
    csv(out(st) / "E8_directions_six.csv", rows)
    return pd.DataFrame(rows)


def lambda_path(st):
    _, m = frozen_models(st)
    base, ep = (m["plans"]["M3"]["h"], m["plans"]["M3"]["epochs"])
    d, nag = (st.data("RICH"), st.data("NAGALA"))
    rows = []
    for lam in LAMBDA_PATH:
        h = dict(base, directions=EXP_DIRS, **{"lambda": float(lam)})
        cfg = st.tune("A", "ext_lambda_path", d, "A_final", grid=[h], epochs=ep)
        model, _ = st.fit("A", "ext_lambda_path", d, None, h, st.seed, ep, "A_ext_lambda_path_refit")
        pr = model.predict(nag)
        g, inside, names = prob_gradients([model], nag)
        viol = {
            f"wrong_way_share_{f}": float(
                np.mean(DIRECTIONS[f] * g[inside[:, names.index(f)], names.index(f)] < 0)
            )
            for f in EXP_DIRS
        }
        rows.append(
            dict(
                lambda_value=lam,
                rich_inner_cv_brier=cfg["score"],
                epochs=ep,
                **metrics(nag, pr, cfg["threshold"]),
                **viol,
            )
        )
    df = pd.DataFrame(rows)
    best = df.rich_inner_cv_brier.min()
    chosen = df[df.rich_inner_cv_brier <= best + 1e-09].lambda_value.min()
    df["selected_by_rich_inner_cv"] = df.lambda_value == chosen
    csv(out(st) / "E9_lambda_path_expanded.csv", df)
    return df


def lambda_path_contrasts(st):
    from stats import bootstrap

    _, m = frozen_models(st)
    base, ep = (m["plans"]["M3"]["h"], m["plans"]["M3"]["epochs"])
    d, nag = (st.data("RICH"), st.data("NAGALA"))
    preds = {}
    for lam in LAMBDA_PATH:
        h = dict(base, directions=EXP_DIRS, **{"lambda": float(lam)})
        model, _ = st.fit("A", "ext_lambda_path", d, None, h, st.seed, ep, "A_ext_lambda_path_refit")
        preds[lam] = model.predict(nag)
    y, w, _ = weights(nag)
    rows = []
    for lam in LAMBDA_PATH[1:]:
        res = bootstrap(nag, preds[lam], preds[0.0], n_boot(st), st.seed + 30)
        rows.append(
            dict(
                comparison=f"lambda {lam:g} minus lambda 0 (six directions)",
                lambda_value=lam,
                delta_brier=brier(y, w, preds[lam]) - brier(y, w, preds[0.0]),
                **res,
            )
        )
    csv(out(st) / "E9b_lambda_path_paired_brier.csv", rows)
    return pd.DataFrame(rows)


def size_curve(st, sizes=None, draws=None):
    _, m = frozen_models(st)
    base = {k: v for k, v in m["plans"]["M3"]["h"].items()}
    rich, nag = (st.data("RICH"), st.data("NAGALA"))
    yn, wn, _ = weights(nag)
    smoke = st.mode == "SMOKE"
    sizes = sizes or ([1000, 2000] if smoke else SIZES)
    draws = draws or ([1] if smoke else DRAWS)
    rows = []
    for size in sizes:
        for draw in draws:
            path = st.private / "ext" / f"size_n{size}_d{draw}.pkl"
            if path.exists():
                rows += restore(path)
                continue
            idx, _ = train_test_split(
                np.arange(len(rich)), train_size=size, stratify=status(rich), random_state=draw
            )
            sub = rich.iloc[np.sort(idx)].reset_index(drop=True)
            label = f"A_ext_n{size}_d{draw}"
            arms = {
                "M1": [dict(base, **{"lambda": 0.0})],
                "penalised_primary": [
                    dict(base, directions=["fpg", "bmi"], **{"lambda": v}) for v in SIZE_LAMBDAS
                ],
                "penalised_expanded": [
                    dict(base, directions=EXP_DIRS, **{"lambda": v}) for v in SIZE_LAMBDAS
                ],
            }
            res = []
            for arm, grid in arms.items():
                cfg = st.tune("A", "ext_size_" + arm, sub, label, grid=grid)
                model, _ = st.fit(
                    "A", "ext_size_" + arm, sub, None, cfg["h"], st.seed, cfg["epochs"], label + "_refit"
                )
                pr = model.predict(nag)
                res.append(
                    dict(
                        size=size,
                        draw=draw,
                        arm=arm,
                        lambda_value=cfg["h"]["lambda"],
                        epochs=cfg["epochs"],
                        cases_in_subsample=int((status(sub) == 1).sum()),
                        nagala_brier=brier(yn, wn, pr),
                        nagala_auroc=metrics(nag, pr, cfg["threshold"])["auroc"],
                        nagala_cal_intercept=calibration(yn, wn, pr)[0],
                    )
                )
            for r in res:
                ref = next((x for x in res if x["arm"] == "M1"))
                r["brier_minus_M1"] = r["nagala_brier"] - ref["nagala_brier"]
                r["auroc_minus_M1"] = r["nagala_auroc"] - ref["nagala_auroc"]
            path.parent.mkdir(parents=True, exist_ok=True)
            dump(path, res)
            rows += res
    df = pd.DataFrame(rows)
    csv(out(st) / "E10_training_size_draws.csv", df)
    summ = (
        df.groupby(["size", "arm"])
        .agg(
            draws=("draw", "size"),
            mean_brier=("nagala_brier", "mean"),
            mean_brier_minus_M1=("brier_minus_M1", "mean"),
            sd_brier_minus_M1=("brier_minus_M1", "std"),
            draws_penalised_better=("brier_minus_M1", lambda v: int((v < 0).sum())),
            mean_auroc=("nagala_auroc", "mean"),
            mean_auroc_minus_M1=("auroc_minus_M1", "mean"),
        )
        .reset_index()
    )
    csv(out(st) / "E10_training_size_summary.csv", summ)
    return df


def without_unsupported_sites(st):
    _, m = frozen_models(st)
    rich, nag = (st.data("RICH"), st.data("NAGALA"))
    sub = rich[~rich.site.isin([14, 16])].reset_index(drop=True)
    rows = []
    for n in ["M1", "M3"]:
        plan = m["plans"][n]
        model, _ = st.fit(
            "A", "ext_nosite_" + n, sub, None, plan["h"], st.seed, plan["epochs"], "A_ext_nosite1416_refit"
        )
        rows.append(
            dict(
                analysis="RICH without sites 14 and 16",
                model=n,
                rich_n=len(sub),
                **metrics(nag, model.predict(nag), plan["threshold"]),
            )
        )
    csv(out(st) / "E11_without_sites_14_16.csv", rows)
    return pd.DataFrame(rows)


def external_subgroups(st):
    d, p, m = st.results("A")
    rows = []
    for model in ["M1", "M3", "LR"]:
        cuts = {
            "all": np.ones(len(d), bool),
            "male": d.sex == 1,
            "female": d.sex == 0,
            "age_20_39": d.age < 40,
            "age_40_59": (d.age >= 40) & (d.age < 60),
            "age_60_plus": d.age >= 60,
            "FPG_below_5.6": d.fpg < 5.6,
            "FPG_5.6_to_6.0": d.fpg >= 5.6,
        }
        for g, mask in cuts.items():
            part = d.loc[np.asarray(mask)].reset_index(drop=True)
            stt = status(part)
            row = dict(
                model=model,
                group=g,
                n=len(part),
                cases=int((stt == 1).sum()),
                controls=int((stt == 0).sum()),
                unknown=int((stt == -1).sum()),
            )
            if row["cases"] < st.cfg["min_cases"] or row["controls"] < st.cfg["min_controls"]:
                row["status"] = "insufficient known cases or controls"
            else:
                row.update(
                    status="estimated",
                    **metrics(part, p[model][np.asarray(mask)], m["plans"][model]["threshold"]),
                )
            rows.append(row)
    csv(out(st) / "EXT_external_subgroups.csv", rows)
    return pd.DataFrame(rows)


def calibration_contrast(st):
    from stats import calbins

    d, p, m = st.results("A")

    def cal(y, w, pr):
        ci, cs = calibration(y, w, pr)
        cb = calbins(y, w, pr)
        return (ci, cs, float(np.average(abs(cb.predicted - cb.observed), weights=cb.n)))

    y, w, _ = weights(d)
    point = {n: cal(y, w, p[n]) for n in ("M1", "M3")}
    rng = np.random.default_rng(st.seed + 41)
    draws, failed, B = ([], 0, n_boot(st))
    for _ in range(B):
        i = rng.integers(len(d), size=len(d))
        dd = d.iloc[i].reset_index(drop=True)
        try:
            yy, ww, _ = weights(dd)
            if np.unique(yy[ww > 0]).size < 2:
                raise ValueError("single class")
            a = {n: cal(yy, ww, p[n][i]) for n in ("M1", "M3")}
            draws.append(
                [
                    abs(a["M3"][0]) - abs(a["M1"][0]),
                    abs(a["M3"][1] - 1) - abs(a["M1"][1] - 1),
                    a["M3"][2] - a["M1"][2],
                ]
            )
        except (ValueError, FloatingPointError):
            failed += 1
    D = np.array(draws, dtype=float)
    labels = ["|calibration intercept|", "|calibration slope - 1|", "ECE"]
    pts = [
        (abs(point["M1"][0]), abs(point["M3"][0])),
        (abs(point["M1"][1] - 1), abs(point["M3"][1] - 1)),
        (point["M1"][2], point["M3"][2]),
    ]
    rows = []
    for k, (lab, (a1, a3)) in enumerate(zip(labels, pts)):
        v = D[:, k][np.isfinite(D[:, k])] if len(D) else np.array([])
        lo, hi = np.quantile(v, [0.025, 0.975]) if len(v) >= max(2, 0.8 * B) else (np.nan, np.nan)
        rows.append(
            dict(
                metric=lab,
                M1=a1,
                M3=a3,
                M3_minus_M1=a3 - a1,
                lo=lo,
                hi=hi,
                valid_boot=len(v),
                requested_boot=B,
                failed_boot=failed,
            )
        )
    csv(out(st) / "E12_calibration_contrast.csv", rows)
    return pd.DataFrame(rows)


def _read(path):
    return pd.read_csv(path, keep_default_na=False, na_values=[""]) if Path(path).exists() else None


def _f(v, d=4):
    try:
        v = float(v)
    except (TypeError, ValueError):
        return str(v)
    if not np.isfinite(v):
        return "NA"
    return f"{v:.3e}" if v != 0 and abs(v) < 0.001 else f"{v:.{d}f}"


def proposal_evidence(st):
    t, e = (st.table / "A", out(st))
    rows = []

    def add(item, question, artifact, estimate, decision="descriptive"):
        rows.append(
            dict(
                item=item, proposal_question=question, artifact=artifact, estimate=estimate, decision=decision
            )
        )

    ex, oof = (_read(t / "external_metrics.csv"), _read(t / "pooled_oof.csv"))
    if ex is not None and oof is not None:
        g = lambda df, n, k: df.set_index("model").loc[n, k]
        add(
            "RQ1",
            "Can a neural network predict future type 2 diabetes from baseline variables?",
            "pooled_oof.csv; external_metrics.csv",
            f"M1 AUROC RICH OOF {_f(g(oof, 'M1', 'auroc'))}, NAGALA {_f(g(ex, 'M1', 'auroc'))}; NAGALA Brier {_f(g(ex, 'M1', 'brier'))} (reference {_f(g(ex, 'reference', 'brier'))}); calibration intercept {_f(g(ex, 'M1', 'cal_intercept'))}, slope {_f(g(ex, 'M1', 'cal_slope'))}",
        )
    pb = _read(t / "paired_brier.csv")
    if pb is not None:
        r = pb.iloc[0]
        dec = "supported" if r.delta_hi < 0 else "contradicted" if r.delta_lo > 0 else "inconclusive"
        add(
            "RQ2 / H1",
            "Does clinical knowledge improve prediction over an unconstrained network (CKINN > MLP)?",
            "paired_brier.csv",
            f"Brier(M3) - Brier(M1) = {_f(r.delta)} (95% {_f(r.delta_lo)} to {_f(r.delta_hi)}); M3 lambda {_f(r.lambda_candidate, 0)}",
            f"{dec} (pre-specified primary contrast)",
        )
    mp = _read(t / "matched_positive_paired_brier.csv")
    if mp is not None:
        r = mp.iloc[0]
        add(
            "RQ2 / H1 (secondary)",
            "Matched positive versus zero penalty, same architecture, seeds and epochs",
            "matched_positive_paired_brier.csv",
            f"Brier difference {_f(r.delta)} (95% {_f(r.delta_lo)} to {_f(r.delta_hi)}); lambda {_f(r.lambda_candidate, 0)}",
        )
    site = _read(t / "A_RICH_site.csv")
    if pb is not None:
        txt = f"NAGALA paired Brier difference {_f(pb.iloc[0].delta)}"
        if site is None:
            txt += "; held-out-site analysis not run"
        else:
            done = site[site.status == "complete"] if "brier" in site else site.iloc[0:0]
            s = (
                done.pivot_table(index="site", columns="model", values="brier")
                if len(done)
                else pd.DataFrame()
            )
            if {"M1", "M3"} <= set(s.columns):
                dd = (s["M3"] - s["M1"]).dropna()
                txt += f"; held-out RICH sites estimable {len(dd)} of {site.site.nunique()}, M3 lower in {int((dd < 0).sum())}, median difference {_f(dd.median())}"
            else:
                txt += f"; no held-out RICH site met the support rule ({site.site.nunique()} sites examined)"
        e9b = _read(e / "E9b_lambda_path_paired_brier.csv")
        if e9b is not None:
            r = e9b.loc[e9b.delta_brier.idxmin()]
            txt += f"; exploratory lambda path (six directions): largest NAGALA reduction {_f(r.delta_brier)} (95% {_f(r.delta_lo)} to {_f(r.delta_hi)}) at lambda {_f(r.lambda_value, 0)}"
        add(
            "RQ3 / H3",
            "Does clinical knowledge improve cross-population generalisation?",
            "paired_brier.csv; A_RICH_site.csv; EXT/E9b",
            txt,
        )
    e12 = _read(e / "E12_calibration_contrast.csv")
    if e12 is not None:
        add(
            "RQ4 / H2",
            "Do clinical constraints improve (maintain) external calibration?",
            "EXT/E12_calibration_contrast.csv",
            "; ".join(
                (
                    f"{r.metric}: M1 {_f(r.M1)}, M3 {_f(r.M3)}, difference {_f(r.M3_minus_M1)} (95% {_f(r.lo)} to {_f(r.hi)})"
                    for r in e12.itertuples()
                )
            ),
        )
    di, e8 = (_read(t / "directions.csv"), _read(e / "E8_directions_six.csv"))
    if di is not None:
        v = di[di.deployment == "frozen_average"].pivot_table(
            index="model", columns="feature", values="violation_rate"
        )
        txt = "; ".join(
            (
                f"{m} wrong-way share FPG {_f(v.loc[m, 'fpg'], 3)}, BMI {_f(v.loc[m, 'bmi'], 3)}"
                for m in ["M1", "M3"]
                if m in v.index
            )
        )
        if e8 is not None:
            w = e8.pivot_table(index="model", columns="feature", values="violation_rate")
            txt += "; " + "; ".join(
                (
                    f"{m} HDL {_f(w.loc[m, 'hdl'], 3)}, SBP {_f(w.loc[m, 'sbp'], 3)}"
                    for m in ["M1", "M3"]
                    if m in w.index
                )
            )
        add(
            "RQ5 / H4",
            "Are CKINN explanations more clinically consistent than an unconstrained network?",
            "directions.csv; attributions.csv; EXT/E8",
            txt,
        )
    sg = _read(t / "sex_gaps.csv")
    if sg is not None:
        parts = []
        for r in sg.itertuples():
            gap = getattr(r, "sensitivity_gap", np.nan)
            parts.append(
                f"{r.model} {r.stage}: {r.status}"
                + (f", sensitivity gap {_f(gap, 3)}" if r.status.startswith("descriptive") else "")
            )
        add(
            "RQ6 / H5",
            "Does the model give more consistent performance across demographic groups?",
            "sex_gaps.csv; subgroups.csv",
            "; ".join(parts),
        )
    sub = _read(t / "subgroups.csv")
    if sub is not None:
        s = sub[sub.group == "FPG_below_5.6"]
        add(
            "RQ7",
            "Can the model identify elevated risk before diagnostic thresholds are reached?",
            "subgroups.csv (FPG < 5.6 mmol/L)",
            "; ".join(
                (
                    f"{r.model} {r.stage}: n {int(r.n)}, cases {int(r.events)}, "
                    + (f"AUROC {_f(r.auroc, 3)}" if r.status == "estimated" else r.status)
                    for r in s.itertuples()
                )
            ),
        )
    for item, q in [
        (
            "Section 3C",
            "Progression from normal through prediabetes to diabetes (repeated dated states required)",
        ),
        ("Section 11", "Latent metabolic state or physiological equations (glucose-insulin data required)"),
        ("Section 24", "Clinical expert agreement with explanations (clinician panel required)"),
        ("Section 26", "Clinical knowledge graph (curated evidence graph required)"),
        ("Section 21", "Clinical decision support and patient benefit (prospective evaluation required)"),
    ]:
        add(item, q, "", "not measurable with these data", "not measurable")
    csv(e / "proposal_hypotheses_evidence.csv", rows)
    return pd.DataFrame(rows)


def run_all(st, include_new_fits=True):
    t = {}
    for name, fn in [
        ("E2", penalty_diagnostics),
        ("E3-E5", external_comparisons),
        ("E6-E7", aligned_and_hdl),
        ("E8", directions_all),
        ("E12", calibration_contrast),
        ("subgroups", external_subgroups),
    ] + (
        [
            ("E9", lambda_path),
            ("E9b", lambda_path_contrasts),
            ("E11", without_unsupported_sites),
            ("E10", size_curve),
        ]
        if include_new_fits
        else []
    ):
        s = time.perf_counter()
        fn(st)
        t[name] = round(time.perf_counter() - s, 1)
        print(f"EXT {name} done in {t[name]} s", flush=True)
    csv(out(st) / "EXT_runtime.csv", [dict(analysis=k, seconds=v) for k, v in t.items()])
    return t

## 12 · Package module `verify_extension.py`: checks of the extension

Hand-calculated net benefit and index of prediction accuracy, plan reconstruction, and an end-to-end run of every extension analysis on artificial records (including resumption from checkpoints).

In [ ]:
%%writefile ckinn_obj1/verify_extension.py
import json, sys, tempfile, unittest
from pathlib import Path
import numpy as np
import pandas as pd

PKG = Path(__file__).resolve().parent
sys.path.insert(0, str(PKG))
import extension as X
from verify import fake
from runner import Study
from stats import weights, brier


class ExtChecks(unittest.TestCase):

    def test_net_benefit_hand(self):
        y = np.array([1.0, 0.0, 0.0, 1.0, 0.0])
        w = np.array([2.0, 1.0, 1.0, 1.0, 0.0])
        p = np.array([0.3, 0.3, 0.01, 0.02, 0.5])
        t = 0.1
        self.assertAlmostEqual(X._nb(y, w, p, t, 5), (2 - t / (1 - t) * 1) / 5)

    def test_ipa_reference(self):
        d = pd.DataFrame(dict(time=[1, 2, 4, 5, 2.5, 6], event=[1, 0, 0, 1, 1, 0]))
        y, w, _ = weights(d)
        ref = np.full(len(d), np.average(y, weights=w))
        self.assertAlmostEqual(1 - brier(y, w, ref) / brier(y, w, ref), 0.0)

    def test_plans_from_selection(self):
        with tempfile.TemporaryDirectory() as td:
            f = Path(td) / "s.csv"
            pd.DataFrame(
                [
                    dict(
                        model="M3",
                        lambda_value=10,
                        source_cv_brier=0.5,
                        threshold=0.0125,
                        candidate_index=2,
                        epochs=17.0,
                        role="",
                        settings=json.dumps({"lambda": 10, "lr": 0.001}),
                    ),
                    dict(
                        model="LR",
                        lambda_value=None,
                        source_cv_brier=0.6,
                        threshold=0.013,
                        candidate_index=1,
                        epochs=None,
                        role="",
                        settings=json.dumps({"C": 0.1}),
                    ),
                ]
            ).to_csv(f, index=False)
            p = X.plans_from_selection(f)
            self.assertEqual(p["M3"]["epochs"], 17)
            self.assertEqual(p["M3"]["h"]["lambda"], 10)
            self.assertEqual(p["LR"]["epochs"], 1)
            self.assertAlmostEqual(p["LR"]["threshold"], 0.013)


def integration():
    cfg = json.loads((PKG / "config.json").read_text())
    cfg.update(profile="SMOKE", stage="ALL", device="cpu")
    cfg["profiles"]["SMOKE"].update(epochs=1, bootstrap=2, trials=1)
    cfg["min_cases"] = cfg["min_controls"] = 3
    cfg["options"]["ablations"] = True
    X.SIZE_LAMBDAS = [10.0]
    X.LAMBDA_PATH = [0.0, 1000.0]
    with tempfile.TemporaryDirectory() as td:
        root = Path(td)
        ident = root / "toy_input.txt"
        ident.write_text("toy records from verify.fake only")
        st = Study(root, cfg, paths={"toy": ident})
        st.mem["RICH"] = fake(3000, "RICH", "A", 7)
        st.mem["NAGALA"] = fake(600, "NAGALA", "A", 14)
        st.internal("A")
        st.freeze("A")
        st.score("A")
        t = X.run_all(st)
        files = sorted((p.name for p in (st.table / "A" / "EXT").glob("*.csv")))
        expected = {
            "E2_penalty_diagnostics.csv",
            "E3_paired_brier_vs_baselines.csv",
            "E4_index_of_prediction_accuracy.csv",
            "E5_net_benefit.csv",
            "E6_estimand_aligned_nagala.csv",
            "E7_nagala_hdl_missing.csv",
            "E8_directions_six.csv",
            "E9_lambda_path_expanded.csv",
            "E10_training_size_draws.csv",
            "E10_training_size_summary.csv",
            "E11_without_sites_14_16.csv",
            "EXT_external_subgroups.csv",
            "EXT_runtime.csv",
        }
        assert expected <= set(files), expected - set(files)
        e9 = pd.read_csv(st.table / "A" / "EXT" / "E9_lambda_path_expanded.csv")
        assert e9.selected_by_rich_inner_cv.sum() == 1
        before = pd.read_csv(st.table / "A" / "EXT" / "E10_training_size_draws.csv")
        X.run_all(st)
        after = pd.read_csv(st.table / "A" / "EXT" / "E10_training_size_draws.csv")
        pd.testing.assert_frame_equal(before, after)
        return f"EXT integration passed on toy records: {len(files)} tables; runtimes {t}"


def run_checks():
    res = unittest.TextTestRunner(verbosity=2).run(
        unittest.defaultTestLoader.loadTestsFromTestCase(ExtChecks)
    )
    if not res.wasSuccessful():
        raise AssertionError("Extension checks failed")
    return {"passed": res.testsRun}


if __name__ == "__main__":
    print(run_checks())
    print(integration())

## 13 · Configuration

Profiles (budgets), the search space, the model list, the options and the support rules. The protocol fixes τ = 3 years and seed 1729.

In [ ]:
%%writefile ckinn_obj1/config.json
{
  "protocol": "Objective1-reanalysis-v2",
  "tau": 3,
  "seed": 1729,
  "profile": "AUDIT",
  "stage": "ALL",
  "final_seed_pool": [
    1729,
    1730,
    1731
  ],
  "profiles": {
    "AUDIT": {
      "outer": 0,
      "inner": 0,
      "trials": 0,
      "epochs": 0,
      "patience": 0,
      "final_seeds": 0,
      "bootstrap": 0,
      "comparator_trials": 0,
      "sample": 0
    },
    "SMOKE": {
      "outer": 2,
      "inner": 2,
      "trials": 2,
      "epochs": 2,
      "patience": 1,
      "final_seeds": 1,
      "bootstrap": 5,
      "comparator_trials": 1,
      "sample": 6000
    },
    "STANDARD": {
      "outer": 5,
      "inner": 2,
      "trials": 6,
      "epochs": 100,
      "patience": 10,
      "final_seeds": 1,
      "bootstrap": 200,
      "comparator_trials": 2,
      "sample": null
    },
    "FULL": {
      "outer": 5,
      "inner": 3,
      "trials": 12,
      "epochs": 100,
      "patience": 10,
      "final_seeds": 3,
      "bootstrap": 1000,
      "comparator_trials": 3,
      "sample": null
    }
  },
  "models": [
    "M1",
    "M3",
    "reference",
    "LR",
    "RF",
    "XGB",
    "LGB",
    "SVM",
    "Cox"
  ],
  "search": {
    "fusion": [
      [
        16,
        8
      ],
      [
        32,
        16
      ],
      [
        64,
        32
      ]
    ],
    "branch": [
      4,
      8,
      16
    ],
    "lr": [
      0.0001,
      0.003
    ],
    "decay": [
      0,
      0.0001,
      0.001
    ],
    "dropout": [
      0,
      0.1,
      0.2
    ],
    "batch": [
      256,
      512,
      1024
    ],
    "lambda": [
      0,
      1,
      10,
      100
    ],
    "comparator_C": [
      1,
      0.1,
      10
    ],
    "ridge": [
      0.1,
      0.01,
      1
    ],
    "depth": [
      3,
      5,
      7
    ],
    "trees": [
      100,
      150,
      200
    ]
  },
  "options": {
    "ablations": true,
    "site_holdout": false,
    "broader_rich": false,
    "alt_zero": false,
    "site_ipcw": true,
    "explain": true,
    "tree_shap": false,
    "subgroups": true,
    "target_update": false,
    "screening": "as_coded",
    "producer_recode_evidence": "",
    "screen_no_code": 0
  },
  "min_cases": 20,
  "min_controls": 20,
  "update_n": 1000,
  "device": "auto"
}

## 14 · Environment check

Checks the installed versions against the declared ranges and installs only missing packages.

In [ ]:
from importlib import metadata
from packaging.requirements import Requirement
import subprocess

REQUIREMENTS = [
    "numpy>=1.26,<3",
    "pandas>=2.1,<3.1",
    "scipy>=1.11,<2",
    "scikit-learn>=1.4,<2",
    "torch>=2.2,<3",
    "lifelines>=0.30,<0.31",
    "lightgbm>=4.1,<5",
    "xgboost>=2,<4",
    "openpyxl>=3.1,<4",
    "matplotlib>=3.7,<4",
    "nbformat>=5.9,<6",
    "shap>=0.45,<1",
]
missing = []
for spec in map(Requirement, REQUIREMENTS):
    try:
        installed = metadata.version(spec.name)
        if installed not in spec.specifier:
            raise RuntimeError(f"{spec.name} {installed} is outside {spec.specifier}")
    except metadata.PackageNotFoundError:
        missing.append(str(spec))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", *missing])
print({spec.name: metadata.version(spec.name) for spec in map(Requirement, REQUIREMENTS)})

## 15 · Imports, seeds and device

Imports the package written above, fixes the seed and enables deterministic PyTorch. LightGBM 4.6 attaches placeholder column names to NumPy inputs and scikit-learn then warns about missing feature names. The warning is suppressed because predictions are identical with and without names.

In [ ]:
import inspect
import warnings

sys.path.insert(0, str(PACKAGE))
warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBMClassifier was fitted with feature names")
from IPython.display import display, Image
import numpy as np
import pandas as pd
from data import discover, js, csv, R, J, CORE, NONLAB, PLUS, UNITS, allowed, digest
from stats import weights, metrics
from models import Prep, clinical, Net, seed_all
from runner import Study, VARIANTS, env_info
from analysis import external_contrast, figures, subgroups, explain, updating, evidence, export, internal_summary
from stages import GROUPS, TRAIN_STAGES, SELECTION_COLS, METRIC_COLS, support_table, show, test_cases
from verify import run_checks
import architecture
import extension
import verify_extension

config = json.loads((PACKAGE / "config.json").read_text())
seed_all(config["seed"])
print(json.dumps(env_info(), indent=2))

## 16 · Run identity, budget and options

Builds the `Study` object, prints the run identifier, budget and options. `active` decides which stages run under `RUN_STAGE`.

In [ ]:
config.update(profile=RUN_MODE, stage=RUN_STAGE, device=DEVICE)
if RUN_MODE not in config["profiles"] or (RUN_STAGE not in ("ALL", "EXT") and RUN_STAGE not in GROUPS):
    raise ValueError("Unknown RUN_MODE or RUN_STAGE")
if set(OPTION_OVERRIDES) - set(config["options"]):
    raise ValueError("Unknown option in OPTION_OVERRIDES")
config["options"].update(OPTION_OVERRIDES)
opt = config["options"]
st = Study(ROOT, config, paths=discover(RAW_DIR))


def active(number):
    selected = RUN_STAGE == "ALL" or (RUN_STAGE in GROUPS and number in GROUPS[RUN_STAGE])
    if selected and RUN_MODE == "AUDIT" and number in TRAIN_STAGES:
        print("Skipped in AUDIT mode.")
        return False
    return selected


def active_ext():
    return RUN_STAGE in ("ALL", "EXT") and RUN_MODE != "AUDIT"


print("Run", st.run, "·", RUN_MODE, "·", st.device)
print(json.dumps({"budget": st.b, "options": opt, "search": config["search"]}, indent=2))

## 17 · Stage 05 · Verify and audit the inputs

Identifies each input by SHA-256, inspects every sheet and ZIP member, and caches the harmonised and eligible tables privately. The first pass takes several minutes.

In [ ]:
if active(5):
    print(pd.Series(st.identity["data"]).to_string())
    st.audit()

## 18 · Stage 06 · Schemas and provenance

Workbook sheets and headers, NHANES archive members, provenance files and the cross-cycle consistency report.

In [ ]:
if active(6):
    info = json.loads((st.auditpath / "inventory.json").read_text(encoding="utf8"))
    for sheet in info["workbooks"]:
        print(sheet)
    print(info["zip_members"], info["consistency_report"])

## 19 · Stage 07 · Completeness, coded values and unit checks

Dimensions, identifiers, duplicate rows, missing and zero values, formula caches and mass-to-molar conversion checks (NAGALA).

In [ ]:
if active(7):
    for name in ["structure", "raw_features", "formula_cache", "unit_checks"]:
        show(st.auditpath / (name + ".csv"))

## 20 · Stage 08 · Harmonised predictor registry

Source headers in both cohorts and the analytical unit of every shared predictor.

In [ ]:
if active(8):
    display(pd.DataFrame([dict(input=k, RICH=R[k], NAGALA=J[k], unit=UNITS[k]) for k in PLUS]))

## 21 · Stage 09 · Eligibility flow

Participants and events after each rule. The FPG < 6.1 mmol/L rule aligns RICH with NAGALA, whose source study excluded higher baseline glucose. It removes 3.4% of RICH participants but more than half of the RICH three-year cases, so it defines the target population as adults with normal fasting glucose. The broader RICH sensitivity analysis keeps 6.1 to < 7.0 mmol/L.

In [ ]:
if active(9):
    show(st.auditpath / "cohort_flow.csv")

## 22 · Stage 10 · Three-year support and censoring

A person with an event after year 3 is a known control. An event-free person last seen at or before year 3 has unknown status. The table reports G(3), weight quantiles, effective sample size and sites without follow-up beyond three years.

Note that RICH participants all have at least two years of follow-up (a source exclusion), so RICH three-year cases are detections in years 2 to 3. EXT E6 evaluates NAGALA under the same condition.

In [ ]:
if active(10):
    display(support_table(st))

## 23 · Stage 11 · Predictor allowlist and development partitions

RICH outer folds are participant-disjoint and stratified by three-year status and site, and are stored before any fit.

In [ ]:
if active(11):
    print("Primary inputs", allowed(CORE, "A"), "Screening inputs", allowed(NONLAB, "B"))
    if st.mode != "AUDIT":
        st.split(st.data("RICH"), st.b["outer"], "A_RICH_nested", st.seed)

## 24 · Stage 12 · Preprocessing learned on training rows

Training medians, missingness indicators, training mean and positive scale, and support limits (1st–99th training percentiles) for the penalty.

In [ ]:
if active(12):
    print(inspect.getsource(Prep))

## 25 · Stage 13 · Censoring weights and known status

A case exactly at year 3 is included. A censoring exactly at year 3 has unknown status.

In [ ]:
if active(13):
    print(inspect.getsource(weights))

## 26 · Stage 14 · Independent checks of the weights and the loss

Seven artificial participants with a tied detection and withdrawal, a detection and a censoring at year 3, and a later detection. The expected non-zero weights are 1, 1.5, 2.25 and 2.25. A separate fixture checks the full-N denominator.

In [ ]:
if active(14):
    test_cases(["test_boundaries_ties_hand_weights", "test_loss_denominator_and_penalty"])

## 27 · Stage 15 · Metrics and calibration

The Brier score is the weighted mean squared error over all eligible participants. Rank measures compare known cases with known controls.

In [ ]:
if active(15):
    print(inspect.getsource(metrics))

## 28 · Stage 16 · Network architecture

Three branches (demographic; body and pressure; laboratory) produce learned features, and a fusion block produces one logit. M1 and M3 share the architecture. M3 adds the clinical penalty during training only.

In [ ]:
if active(16):
    architecture.draw(st.fig / "architecture.png")
    display(Image(filename=str(st.fig / "architecture.png")))

## 29 · Stage 17 · The soft clinical penalty

The penalty uses the local probability derivative with respect to each constrained input. Increasing FPG and BMI have positive priors in task A. Only observed, in-support pairs contribute.

In [ ]:
if active(17):
    test_cases(["test_loss_denominator_and_penalty", "test_derivative_sign_masks_and_raw_finite_difference"])
    print(inspect.getsource(clinical))

## 30 · Stage 18 · Statistical comparators

Logistic regression, random forest, XGBoost, LightGBM and a linear SVM are fitted with IPCW sample weights, with the SVM calibrated on training-only cross-fitted scores. Ridge Cox is administratively truncated at three years. The reference model predicts the weighted training event fraction.

In [ ]:
if active(18):
    print(st.cfg["models"], "Comparator candidate budget", st.b["comparator_trials"])

## 31 · Stage 19 · Tuning, stopping and checkpoints

Each inner fold learns its own preprocessing and censoring weights. Candidates are selected by validation Brier score, and models are refitted on the outer training partition for the median selected epoch count.

In [ ]:
if active(19):
    print(st.b, "Checkpoint directory", st.private)

## 32 · Stage 20 · Software checks

Runs the unit checks of the package and of the extension, on artificial arrays only.

In [ ]:
if active(20):
    js(st.report / "software_checks.json", {"package": run_checks(), "extension": verify_extension.run_checks()})

## 33 · Stage 21 · Nested internal validation in RICH

Within each outer training fold, every model is tuned in inner folds. The held-out outer participants are scored once. A site-specific censoring model is fitted as a sensitivity analysis.

In [ ]:
if active(21):
    display(st.internal().head())

## 34 · Stage 22 · Internal results

Fold-level results and pooled out-of-fold metrics. Folds share training participants and are not independent replicates.

In [ ]:
if active(22):
    show(st.table / "A" / "A_RICH_nested.csv")
    display(internal_summary(st))

## 35 · Stage 23 · Final configurations selected in RICH

The inner procedure is repeated on all eligible RICH participants. The matched pair is anchored on the M3 settings and epoch count; its positive λ (1, 10 or 100) is chosen on the RICH inner folds only.

In [ ]:
if active(23):
    display(st.selection_table("A")[SELECTION_COLS])

## 36 · Stage 24 · Final fits and freeze manifest

The selected configurations are fitted on RICH. Models, preprocessing and hashes enter the freeze manifest before any external scoring.

In [ ]:
if active(24):
    print(st.freeze("A"))

## 37 · Stage 25 · Frozen scoring in NAGALA

The frozen pipelines are applied unchanged to NAGALA.

In [ ]:
if active(25):
    display(st.score("A")[METRIC_COLS])

## 38 · Stage 26 · Paired external contrasts (H1)

Brier(M3) − Brier(M1) and Brier(matched positive) − Brier(matched zero) in the same NAGALA participants. Negative values favour the constrained model. A zero-crossing interval is a valid result.

In [ ]:
if active(26):
    display(pd.Series(external_contrast(st)))
    if opt["ablations"]:
        display(pd.Series(external_contrast(st, matched=True)))

## 39 · Stage 27 · Calibration, ROC and precision–recall in NAGALA

Decile calibration against the IPCW event fraction, and ROC and precision–recall curves, with the threshold chosen on RICH validation predictions.

In [ ]:
if active(27):
    figures(st, "A")
    show(st.fig / "A" / "calibration_roc_pr.png")

## 40 · Stage 28 · M1–M5 and component variants

M2 reuses M1 predictions. M4 adds training site weights, and M5 a sex loss-gap penalty. The FPG-only, BMI-only and expanded-prior variants use the matched positive λ. Also included: no masks, single stream, expanded inputs and the four-input non-laboratory network.

In [ ]:
if active(28):
    if opt["ablations"]:
        table = pd.read_csv(st.table / "A" / "external_metrics.csv")
        if not set(VARIANTS).issubset(table.model):
            raise ValueError("Component fits missing from the frozen manifest")
        csv(st.table / "A" / "ablations.csv", table[table.model.isin(VARIANTS + ["M1", "M2", "M3"])])
        display(pd.read_csv(st.table / "A" / "ablations.csv")[METRIC_COLS])

## 41 · Stage 29 · Sensitivity analyses (H3)

* Leave-one-site-out validation in RICH (sites with fewer than 20 cases or controls are reported as unsupported).
* Broader RICH eligibility (FPG < 7.0 mmol/L).
* ALT zeros retained.

This is the longest stage, and it resumes after interruption.

In [ ]:
if active(29):
    for option, source, site in [("site_holdout", "RICH", True), ("broader_rich", "RICH_broad", False), ("alt_zero", "RICH_alt0", False)]:
        if opt[option]:
            display(st.internal(source=source, site=site).head())
        else:
            print("Disabled", option)
    show(st.auditpath / "feature_set_availability.csv")

## 42 · Stage 30 · Explanations and clinical direction (H4)

* Integrated gradients from training medians, with a completeness check.
* Finite raw-unit steps and analytical gradients for FPG and BMI (all six expanded directions for the expanded-prior model).
* ICE curves.
* Random-forest probability SHAP.

These describe model behaviour, not causal effects.

In [ ]:
if active(30):
    if opt["explain"]:
        explain(st, "A")
        show(st.table / "A" / "directions.csv")
        show(st.table / "A" / "attributions.csv")

## 43 · Stage 31 · Subgroups (H5, RQ7)

Sex, age bands, baseline FPG < 5.6 mmol/L, predictor availability and (in RICH out-of-fold predictions) site, for M1, M3, M4 and M5. Groups need at least 20 known cases and 20 known controls for estimates.

In [ ]:
if active(31):
    if opt["subgroups"]:
        display(subgroups(st, "A").head())

## 44 · Stage 32 · Optional target intercept update

Disabled by default. If enabled, 1,000 NAGALA participants update the intercept and the remaining participants evaluate it, after the frozen results have been reported.

In [ ]:
if active(32):
    if opt["target_update"]:
        display(pd.DataFrame(updating(st)))
    else:
        print("Target updating disabled.")

## 45 · Stage 33 · NHANES screening eligibility (task B)

Adults with measured HbA1c and the extract's no-diagnosis code. A screen is positive at HbA1c ≥ 6.5%. The extract's recoding of borderline and unknown responses is undocumented, so results are labelled as coded.

In [ ]:
if active(33):
    show(st.auditpath / "cohort_flow.csv")
    print("NHANES analysis label", st.tag("B"), "| producer recode evidence:", opt["producer_recode_evidence"] or "unavailable")

## 46 · Stage 34 · Development and freeze in NHANES 2017–2020

Inputs are age, sex, BMI and systolic blood pressure, with a BMI-only positive prior. Glycaemic measurements and self-reported diabetes never enter the models.

In [ ]:
if active(34):
    if opt["screening"] != "off":
        print(st.freeze("B"))
        display(st.selection_table("B", frozen=True)[SELECTION_COLS])

## 47 · Stage 35 · Frozen evaluation in NHANES 2021–2023

Screening metrics, the paired contrast, calibration, subgroups and direction checks in the later period. The extract has no survey design variables, so estimates apply to the extract only.

In [ ]:
if active(35):
    if opt["screening"] != "off":
        display(st.score("B")[METRIC_COLS])
        display(pd.Series(external_contrast(st, "B")))
        figures(st, "B")
        if opt["subgroups"]:
            subgroups(st, "B")
        if opt["explain"]:
            explain(st, "B")

## 48 · EXT · Extension analyses (exploratory)

The rules for E2 to E12 are in `EXT_PRESPECIFICATION.md`. New fits (E9, E10, E11) are checkpointed like all other fits. Outputs are in `results/tables/<mode>/<run>/A/EXT/`.

In [ ]:
if active_ext():
    print(extension.run_all(st))
    for name in ["E2_penalty_diagnostics.csv", "E3_paired_brier_vs_baselines.csv", "E9_lambda_path_expanded.csv",
                 "E9b_lambda_path_paired_brier.csv", "E10_training_size_summary.csv", "E12_calibration_contrast.csv"]:
        show(st.table / "A" / "EXT" / name)

## 49 · Stage 36 · Inventory of tables and fit budget

Lists every aggregate table and summarises the recorded fits and run times.

In [ ]:
if active(36):
    for path in sorted(st.table.rglob("*.csv")):
        print(path.relative_to(st.table))
    if (st.log / "fits.jsonl").exists():
        fits = pd.read_json(st.log / "fits.jsonl", lines=True)
        csv(st.table / "fit_budget.csv",
            fits.groupby(["task", "model", "status"], dropna=False).agg(fits=("model", "size"), seconds=("seconds", "sum")).reset_index())

## 50 · Stage 37 · Proposal questions, hypotheses and the claim register

Maps RQ1–RQ7 and H1–H5 of the proposal to the computed estimates. Only H1 has a decision rule. It also writes the claim-to-evidence register and a narrative generated from the saved tables.

In [ ]:
if active(37):
    display(extension.proposal_evidence(st))
    evidence(st)
    show(st.report / "summary.md")

## 51 · Stage 38 · Reproducibility manifest

Input and code identities, environment, settings, budgets, available tables and frozen-model hashes.

In [ ]:
if active(38):
    evidence(st)
    show(st.report / "run_manifest.json")

## 52 · Stage 39 · Aggregate export

Writes a ZIP of aggregate tables, figures, reports, the package code and logs. Models, participant-level predictions, individual explanations and source data stay in `checkpoints/`.

In [ ]:
if active(39):
    print(export(st, PACKAGE))